# Week 8 — Deployment and Final Analysis

## Student Skill Gap Analysis and Career Recommendation System Using NLP and Recommendation Systems

This notebook integrates the completed recommendation, prescriptive analytics, labour-demand, education-alignment, and RAG components into deployment-ready outputs.

### Week 8 objectives

1. Load and validate final Week 6 and Week 7 outputs
2. Build deployment-ready candidate recommendation tables
3. Integrate Top-5 career recommendations
4. Integrate prescriptive skill-gap recommendations
5. Integrate education and career-action guidance
6. Integrate grounded RAG explanations
7. Produce Streamlit-ready datasets
8. Generate final candidate and occupation analytics
9. Produce final project KPIs and business insights
10. Validate deployment artifacts and reproducibility

### Evaluation note

Human-ground-truth evaluation from Week 7 remains pending until manual review is completed. No human-dependent Precision, Recall, F1, ranking, skill-gap, or RAG quality metrics are reported until those labels have been reviewed.

In [1]:
# ============================================================
# WEEK 8 — CELL 1
# IMPORT LIBRARIES AND DEFINE PROJECT PATHS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import json
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\Admin\Capstone_Project"
)

OUTPUT_DIR = PROJECT_ROOT / "Outputs"
WEEK6_OUTPUT_DIR = OUTPUT_DIR / "Tables" / "Week6"
WEEK7_OUTPUT_DIR = OUTPUT_DIR / "Tables" / "Week7"
WEEK8_OUTPUT_DIR = OUTPUT_DIR / "Tables" / "Week8"

# Create Week 8 output directory
WEEK8_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — PROJECT PATH SETUP")
print("=" * 70)

print(f"\nProject root:")
print(PROJECT_ROOT)

print(f"\nWeek 6 output directory:")
print(WEEK6_OUTPUT_DIR)

print(f"\nWeek 7 output directory:")
print(WEEK7_OUTPUT_DIR)

print(f"\nWeek 8 output directory:")
print(WEEK8_OUTPUT_DIR)

print("\nDirectory checks:")
print("Project root exists :", PROJECT_ROOT.exists())
print("Week 6 exists       :", WEEK6_OUTPUT_DIR.exists())
print("Week 7 exists       :", WEEK7_OUTPUT_DIR.exists())
print("Week 8 exists       :", WEEK8_OUTPUT_DIR.exists())

print("\n✅ Week 8 environment initialized.")

WEEK 8 — PROJECT PATH SETUP

Project root:
C:\Users\Admin\Capstone_Project

Week 6 output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week6

Week 7 output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week7

Week 8 output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week8

Directory checks:
Project root exists : True
Week 6 exists       : True
Week 7 exists       : True
Week 8 exists       : True

✅ Week 8 environment initialized.


In [3]:
# ============================================================
# WEEK 8 — CELL 2
# CHECK REQUIRED WEEK 6 AND WEEK 7 INPUT FILES
# ============================================================

# ------------------------------------------------------------
# Required Week 6 files
# ------------------------------------------------------------

week6_required_files = [
    "week6_final_hybrid_recommendations.csv",
]

# ------------------------------------------------------------
# Required Week 7 files
# ------------------------------------------------------------

week7_required_files = [
    "week7_missing_skill_priorities.csv",
    "week7_candidate_prescriptive_recommendations.csv",
    "week7_top1_prescriptive_skill_gaps.csv",
    "week7_final_prescriptive_recommendations.csv",
    "week7_rag_knowledge_base.csv",
    "week7_candidate_rag_explanations.csv",
    "week7_ablation_contribution_summary.csv",
    "week7_project_limitations.csv",
]

# ------------------------------------------------------------
# Check file existence
# ------------------------------------------------------------

file_check_rows = []

for filename in week6_required_files:
    path = WEEK6_OUTPUT_DIR / filename

    file_check_rows.append({
        "week": "Week6",
        "filename": filename,
        "exists": path.exists(),
        "path": str(path)
    })

for filename in week7_required_files:
    path = WEEK7_OUTPUT_DIR / filename

    file_check_rows.append({
        "week": "Week7",
        "filename": filename,
        "exists": path.exists(),
        "path": str(path)
    })

week8_input_file_check = pd.DataFrame(file_check_rows)

print("=" * 70)
print("WEEK 8 — INPUT FILE CHECK")
print("=" * 70)

display(
    week8_input_file_check[
        ["week", "filename", "exists"]
    ]
)

missing_files = week8_input_file_check[
    ~week8_input_file_check["exists"]
]

print("\nSummary:")
print("Required files :", len(week8_input_file_check))
print("Files found    :", week8_input_file_check["exists"].sum())
print("Files missing  :", len(missing_files))

if len(missing_files) == 0:
    print("\n✅ All required Week 8 input files are available.")
else:
    print("\n⚠️ Some required files are missing:")
    display(missing_files[["week", "filename", "path"]])

WEEK 8 — INPUT FILE CHECK


,week,filename,exists
0,Week6,week6_final_hybrid_recommendations.csv,True
1,Week7,week7_missing_skill_priorities.csv,True
2,Week7,week7_candidate_prescriptive_recommendations.csv,True
3,Week7,week7_top1_prescriptive_skill_gaps.csv,True
4,Week7,week7_final_prescriptive_recommendations.csv,True
5,Week7,week7_rag_knowledge_base.csv,True
6,Week7,week7_candidate_rag_explanations.csv,True
7,Week7,week7_ablation_contribution_summary.csv,True
8,Week7,week7_project_limitations.csv,True



Summary:
Required files : 9
Files found    : 9
Files missing  : 0

✅ All required Week 8 input files are available.


In [5]:
# ============================================================
# WEEK 8 — CELL 3
# LOAD WEEK 6 AND WEEK 7 INPUT DATASETS
# ============================================================

# ------------------------------------------------------------
# Load Week 6
# ------------------------------------------------------------

final_hybrid = pd.read_csv(
    WEEK6_OUTPUT_DIR / "week6_final_hybrid_recommendations.csv"
)

# ------------------------------------------------------------
# Load Week 7
# ------------------------------------------------------------

missing_skill_priorities = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_missing_skill_priorities.csv"
)

candidate_prescriptive_recommendations = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_candidate_prescriptive_recommendations.csv"
)

top1_prescriptive_skill_gaps = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_top1_prescriptive_skill_gaps.csv"
)

final_prescriptive_recommendations = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_final_prescriptive_recommendations.csv"
)

rag_knowledge_base = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_rag_knowledge_base.csv"
)

candidate_rag_explanations = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_candidate_rag_explanations.csv"
)

ablation_contribution_summary = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_ablation_contribution_summary.csv"
)

project_limitations = pd.read_csv(
    WEEK7_OUTPUT_DIR / "week7_project_limitations.csv"
)

# ------------------------------------------------------------
# Display dataset shapes
# ------------------------------------------------------------

dataset_shapes = pd.DataFrame([
    ["final_hybrid", *final_hybrid.shape],
    ["missing_skill_priorities", *missing_skill_priorities.shape],
    ["candidate_prescriptive_recommendations", *candidate_prescriptive_recommendations.shape],
    ["top1_prescriptive_skill_gaps", *top1_prescriptive_skill_gaps.shape],
    ["final_prescriptive_recommendations", *final_prescriptive_recommendations.shape],
    ["rag_knowledge_base", *rag_knowledge_base.shape],
    ["candidate_rag_explanations", *candidate_rag_explanations.shape],
    ["ablation_contribution_summary", *ablation_contribution_summary.shape],
    ["project_limitations", *project_limitations.shape],
], columns=["dataset", "rows", "columns"])

print("=" * 70)
print("WEEK 8 — DATASET LOAD SUMMARY")
print("=" * 70)

display(dataset_shapes)

print("\nKey checks:")
print("Candidates in final_hybrid        :", final_hybrid["candidate_id"].nunique())
print("Occupations in final_hybrid       :", final_hybrid["selected_occupation"].nunique())
print("Candidate-occupation pairs        :", len(final_hybrid))
print("RAG knowledge documents           :", len(rag_knowledge_base))
print("Candidate RAG explanations        :", len(candidate_rag_explanations))

print("\n✅ Week 6 and Week 7 datasets loaded.")

WEEK 8 — DATASET LOAD SUMMARY


,dataset,rows,columns
0,final_hybrid,945,31
1,missing_skill_priorities,2610,19
2,candidate_prescriptive_recommendations,2070,18
3,top1_prescriptive_skill_gaps,138,21
4,final_prescriptive_recommendations,56,11
5,rag_knowledge_base,707,6
6,candidate_rag_explanations,63,14
7,ablation_contribution_summary,4,9
8,project_limitations,14,4



Key checks:
Candidates in final_hybrid        : 63
Occupations in final_hybrid       : 15
Candidate-occupation pairs        : 945
RAG knowledge documents           : 707
Candidate RAG explanations        : 63

✅ Week 6 and Week 7 datasets loaded.


In [7]:
# ============================================================
# WEEK 8 — CELL 4
# BUILD TOP-5 DEPLOYMENT RECOMMENDATIONS
# ============================================================

# ------------------------------------------------------------
# Select Top 5 occupations per candidate
# ------------------------------------------------------------

top5_recommendations = (
    final_hybrid
    .sort_values(
        ["candidate_id", "final_hybrid_rank"]
    )
    .groupby("candidate_id", as_index=False)
    .head(5)
    .copy()
)

# ------------------------------------------------------------
# Keep deployment-relevant columns
# ------------------------------------------------------------

top5_recommendations = top5_recommendations[
    [
        "candidate_id",
        "selected_occupation",
        "final_hybrid_rank",
        "final_hybrid_score",
        "final_hybrid_percentage",
        "skill_percentage",
        "semantic_percentage",
        "demand_percentage",
        "education_percentage"
    ]
].copy()

# ------------------------------------------------------------
# Rename for deployment readability
# ------------------------------------------------------------

top5_recommendations = top5_recommendations.rename(
    columns={
        "selected_occupation": "recommended_occupation",
        "final_hybrid_rank": "recommendation_rank",
        "final_hybrid_score": "recommendation_score",
        "final_hybrid_percentage": "recommendation_percentage"
    }
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

candidate_counts = (
    top5_recommendations
    .groupby("candidate_id")
    .size()
)

print("=" * 70)
print("WEEK 8 — TOP-5 RECOMMENDATIONS")
print("=" * 70)

print("\nShape:")
print(top5_recommendations.shape)

print("\nCandidates:")
print(top5_recommendations["candidate_id"].nunique())

print("\nRows per candidate:")
print(candidate_counts.value_counts().sort_index())

print("\nRank values:")
print(sorted(top5_recommendations["recommendation_rank"].unique()))

print("\nSample:")
display(top5_recommendations.head(10))

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(top5_recommendations) == 63 * 5
assert top5_recommendations["candidate_id"].nunique() == 63
assert set(top5_recommendations["recommendation_rank"].unique()) == {1, 2, 3, 4, 5}

print("\n✅ Top-5 deployment recommendation table created successfully.")

WEEK 8 — TOP-5 RECOMMENDATIONS

Shape:
(315, 9)

Candidates:
63

Rows per candidate:
5    63
Name: count, dtype: int64

Rank values:
[1, 2, 3, 4, 5]

Sample:


,candidate_id,recommended_occupation,recommendation_rank,recommendation_score,recommendation_percentage,skill_percentage,semantic_percentage,demand_percentage,education_percentage
0,Candidate_001,Software Developer,1,0.845817,84.58,100.00,100.00,22.91,100.0
1,Candidate_001,Information Technology (IT) Analyst,2,0.652990,65.30,69.68,73.10,26.64,100.0
2,Candidate_001,Bookkeeper,3,0.603429,60.34,61.28,65.46,29.92,100.0
3,Candidate_001,Secondary School Teacher,4,0.591983,59.20,62.29,57.50,36.36,100.0
4,Candidate_001,Office Administrator,5,0.465784,46.58,43.62,59.26,52.86,0.0
15,Candidate_002,Software Developer,1,0.782459,78.25,81.90,100.00,22.91,100.0
16,Candidate_002,Office Manager,2,0.671262,67.13,100.00,33.01,52.86,100.0
17,Candidate_002,Information Technology (IT) Analyst,3,0.634155,63.42,61.55,75.84,26.64,100.0
18,Candidate_002,Office Administrator,4,0.622124,62.21,81.98,36.99,52.86,100.0
19,Candidate_002,Secondary School Teacher,5,0.610059,61.01,78.23,46.73,36.36,100.0



✅ Top-5 deployment recommendation table created successfully.


In [9]:
# ============================================================
# WEEK 8 — CELL 5
# BUILD TOP-1 CANDIDATE RECOMMENDATION SUMMARY
# ============================================================

# ------------------------------------------------------------
# Select the best recommendation for each candidate
# ------------------------------------------------------------

top1_candidate_summary = (
    top5_recommendations[
        top5_recommendations["recommendation_rank"] == 1
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Rename columns for final deployment readability
# ------------------------------------------------------------

top1_candidate_summary = top1_candidate_summary.rename(
    columns={
        "recommended_occupation": "top_recommended_occupation",
        "recommendation_score": "top_recommendation_score",
        "recommendation_percentage": "top_recommendation_percentage",
        "skill_percentage": "top_skill_percentage",
        "semantic_percentage": "top_semantic_percentage",
        "demand_percentage": "top_demand_percentage",
        "education_percentage": "top_education_percentage"
    }
)

# Recommendation rank is always 1 here, so remove it
top1_candidate_summary = top1_candidate_summary.drop(
    columns=["recommendation_rank"]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — TOP-1 CANDIDATE SUMMARY")
print("=" * 70)

print("\nShape:")
print(top1_candidate_summary.shape)

print("\nUnique candidates:")
print(top1_candidate_summary["candidate_id"].nunique())

print("\nDuplicate candidate IDs:")
print(top1_candidate_summary["candidate_id"].duplicated().sum())

print("\nTop recommended occupation distribution:")
display(
    top1_candidate_summary[
        "top_recommended_occupation"
    ]
    .value_counts()
    .rename_axis("occupation")
    .reset_index(name="candidate_count")
)

print("\nSample:")
display(top1_candidate_summary.head(10))

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(top1_candidate_summary) == 63
assert top1_candidate_summary["candidate_id"].nunique() == 63
assert top1_candidate_summary["candidate_id"].duplicated().sum() == 0

print("\n✅ Top-1 candidate recommendation summary created successfully.")

WEEK 8 — TOP-1 CANDIDATE SUMMARY

Shape:
(63, 8)

Unique candidates:
63

Duplicate candidate IDs:
0

Top recommended occupation distribution:


,occupation,candidate_count
0,Software Developer,35
1,Administrative Assistant,13
2,Food Service Supervisor,11
3,Office Manager,4



Sample:


,candidate_id,top_recommended_occupation,top_recommendation_score,top_recommendation_percentage,top_skill_percentage,top_semantic_percentage,top_demand_percentage,top_education_percentage
0,Candidate_001,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0
1,Candidate_002,Software Developer,0.782459,78.25,81.90,100.00,22.91,100.0
2,Candidate_003,Administrative Assistant,0.743337,74.33,100.00,58.76,43.85,100.0
3,Candidate_004,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0
4,Candidate_005,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0
5,Candidate_006,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0
6,Candidate_007,Food Service Supervisor,0.743113,74.31,100.00,44.70,68.33,100.0
7,Candidate_008,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0
8,Candidate_009,Administrative Assistant,0.741840,74.18,100.00,58.33,43.85,100.0
9,Candidate_010,Food Service Supervisor,0.721099,72.11,92.66,45.75,68.33,100.0



✅ Top-1 candidate recommendation summary created successfully.


In [11]:
# ============================================================
# WEEK 8 — CELL 6
# INTEGRATE TOP-1 RECOMMENDATIONS WITH PRESCRIPTIVE SKILL GAPS
# ============================================================

# ------------------------------------------------------------
# Inspect available prescriptive columns
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — PRESCRIPTIVE INTEGRATION")
print("=" * 70)

print("\nFinal prescriptive recommendation columns:")
print(final_prescriptive_recommendations.columns.tolist())

print("\nShape:")
print(final_prescriptive_recommendations.shape)

print("\nSample:")
display(final_prescriptive_recommendations.head())

# ------------------------------------------------------------
# Merge Top-1 recommendation summary with Week 7 prescriptive output
# ------------------------------------------------------------

top1_with_prescriptive = top1_candidate_summary.merge(
    final_prescriptive_recommendations,
    on="candidate_id",
    how="left",
    suffixes=("", "_prescriptive")
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nMerged shape:")
print(top1_with_prescriptive.shape)

print("\nUnique candidates:")
print(top1_with_prescriptive["candidate_id"].nunique())

print("\nCandidates with prescriptive information:")
print(
    top1_with_prescriptive[
        final_prescriptive_recommendations.columns[
            final_prescriptive_recommendations.columns != "candidate_id"
        ][0]
    ].notna().sum()
)

print("\nCandidates without prescriptive information:")
print(
    top1_with_prescriptive[
        final_prescriptive_recommendations.columns[
            final_prescriptive_recommendations.columns != "candidate_id"
        ][0]
    ].isna().sum()
)

print("\nSample merged rows:")
display(top1_with_prescriptive.head(10))

assert len(top1_with_prescriptive) == 63
assert top1_with_prescriptive["candidate_id"].nunique() == 63

print("\n✅ Prescriptive skill-gap information integrated successfully.")

WEEK 8 — PRESCRIPTIVE INTEGRATION

Final prescriptive recommendation columns:
['candidate_id', 'recommended_occupation', 'hybrid_recommendation_score', 'recommendation_explanation', 'prioritized_skill_gaps', 'average_prescriptive_priority', 'maximum_prescriptive_priority', 'demand_percentage', 'demand_level', 'cip_status', 'education_action']

Shape:
(56, 11)

Sample:


,candidate_id,recommended_occupation,hybrid_recommendation_score,recommendation_explanation,prioritized_skill_gaps,average_prescriptive_priority,maximum_prescriptive_priority,demand_percentage,demand_level,cip_status,education_action
0,Candidate_001,Software Developer,69.05,Moderate recommendation based on combined cand...,Active Listening (52.72%); Speaking (49.60%); ...,50.16,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
1,Candidate_002,Office Manager,55.62,Moderate recommendation based on combined cand...,Monitoring (69.14%); Speaking (69.14%); Active...,68.90,69.14,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways
2,Candidate_003,Administrative Assistant,93.24,Strong match based on both baseline candidate ...,Mathematics (39.26%); Science (23.54%),31.40,39.26,43.85,Moderate,CIP pathway available,Review mapped CIP education pathways
3,Candidate_004,Software Developer,78.59,Strong match based on both baseline candidate ...,Active Listening (52.72%); Speaking (49.60%),51.16,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
4,Candidate_005,Software Developer,53.35,Moderate recommendation based on combined cand...,Active Listening (52.72%); Writing (50.38%); S...,50.90,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways



Merged shape:
(63, 18)

Unique candidates:
63

Candidates with prescriptive information:
56

Candidates without prescriptive information:
7

Sample merged rows:


,candidate_id,top_recommended_occupation,top_recommendation_score,top_recommendation_percentage,top_skill_percentage,top_semantic_percentage,top_demand_percentage,top_education_percentage,recommended_occupation,hybrid_recommendation_score,recommendation_explanation,prioritized_skill_gaps,average_prescriptive_priority,maximum_prescriptive_priority,demand_percentage,demand_level,cip_status,education_action
0,Candidate_001,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,Software Developer,69.05,Moderate recommendation based on combined cand...,Active Listening (52.72%); Speaking (49.60%); ...,50.16,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
1,Candidate_002,Software Developer,0.782459,78.25,81.90,100.00,22.91,100.0,Office Manager,55.62,Moderate recommendation based on combined cand...,Monitoring (69.14%); Speaking (69.14%); Active...,68.90,69.14,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways
2,Candidate_003,Administrative Assistant,0.743337,74.33,100.00,58.76,43.85,100.0,Administrative Assistant,93.24,Strong match based on both baseline candidate ...,Mathematics (39.26%); Science (23.54%),31.40,39.26,43.85,Moderate,CIP pathway available,Review mapped CIP education pathways
3,Candidate_004,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,Software Developer,78.59,Strong match based on both baseline candidate ...,Active Listening (52.72%); Speaking (49.60%),51.16,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
4,Candidate_005,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,Software Developer,53.35,Moderate recommendation based on combined cand...,Active Listening (52.72%); Writing (50.38%); S...,50.90,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
5,Candidate_006,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,Software Developer,90.46,Strong match based on both baseline candidate ...,Monitoring (48.16%),48.16,48.16,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
6,Candidate_007,Food Service Supervisor,0.743113,74.31,100.00,44.70,68.33,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Candidate_008,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,Software Developer,68.08,Moderate recommendation based on combined cand...,Active Listening (52.72%); Writing (50.38%); S...,50.90,52.72,22.91,Lower,CIP pathway available,Review mapped CIP education pathways
8,Candidate_009,Administrative Assistant,0.741840,74.18,100.00,58.33,43.85,100.0,Administrative Assistant,27.73,Moderate recommendation based on combined cand...,Active Listening (64.04%); Speaking (63.26%); ...,61.52,64.04,43.85,Moderate,CIP pathway available,Review mapped CIP education pathways
9,Candidate_010,Food Service Supervisor,0.721099,72.11,92.66,45.75,68.33,100.0,Office Manager,98.56,Strong match based on both baseline candidate ...,Science (27.14%),27.14,27.14,52.86,Moderate,CIP pathway available,Review mapped CIP education pathways



✅ Prescriptive skill-gap information integrated successfully.


In [13]:
# ============================================================
# WEEK 8 — CELL 6A
# INSPECT TOP-1 PRESCRIPTIVE SKILL-GAP TABLE
# ============================================================

print("=" * 70)
print("WEEK 8 — TOP-1 PRESCRIPTIVE TABLE INSPECTION")
print("=" * 70)

print("\nShape:")
print(top1_prescriptive_skill_gaps.shape)

print("\nColumns:")
print(top1_prescriptive_skill_gaps.columns.tolist())

print("\nUnique candidates:")
print(top1_prescriptive_skill_gaps["candidate_id"].nunique())

print("\nUnique occupations:")
print(top1_prescriptive_skill_gaps["selected_occupation"].nunique())

print("\nSample:")
display(top1_prescriptive_skill_gaps.head(10))

WEEK 8 — TOP-1 PRESCRIPTIVE TABLE INSPECTION

Shape:
(138, 21)

Columns:
['candidate_id', 'selected_occupation', 'skill', 'importance', 'level', 'candidate_evidence', 'skill_gap_priority', 'demand_percentage', 'prescriptive_priority_score', 'prescriptive_priority_percentage', 'skill_priority_rank', 'cip_program_count', 'cip_pathway_available', 'cip_status', 'education_action', 'priority_level', 'demand_level', 'prescriptive_action', 'recommended_occupation', 'hybrid_recommendation_score', 'recommendation_explanation']

Unique candidates:
56

Unique occupations:
6

Sample:


,candidate_id,selected_occupation,skill,importance,level,candidate_evidence,skill_gap_priority,demand_percentage,prescriptive_priority_score,prescriptive_priority_percentage,...,cip_program_count,cip_pathway_available,cip_status,education_action,priority_level,demand_level,prescriptive_action,recommended_occupation,hybrid_recommendation_score,recommendation_explanation
0,Candidate_001,Software Developer,Active Listening,3.38,3.88,0.0,0.726,22.91,0.527,52.72,...,20,True,CIP pathway available,Review mapped CIP education pathways,Medium,Lower,Consider developing Active Listening as a medi...,Software Developer,69.054471,Moderate recommendation based on combined cand...
1,Candidate_001,Software Developer,Speaking,3.12,3.62,0.0,0.674,22.91,0.496,49.60,...,20,True,CIP pathway available,Review mapped CIP education pathways,Lower,Lower,Consider developing Speaking as an additional ...,Software Developer,69.054471,Moderate recommendation based on combined cand...
2,Candidate_001,Software Developer,Monitoring,3.00,3.50,0.0,0.650,22.91,0.482,48.16,...,20,True,CIP pathway available,Review mapped CIP education pathways,Lower,Lower,Consider developing Monitoring as an additiona...,Software Developer,69.054471,Moderate recommendation based on combined cand...
3,Candidate_002,Office Manager,Monitoring,4.00,4.00,0.0,0.800,52.86,0.691,69.14,...,13,True,CIP pathway available,Review mapped CIP education pathways,Medium,Moderate,Consider developing Monitoring as a medium-pri...,Office Manager,55.624220,Moderate recommendation based on combined cand...
4,Candidate_002,Office Manager,Speaking,4.00,4.00,0.0,0.800,52.86,0.691,69.14,...,13,True,CIP pathway available,Review mapped CIP education pathways,Medium,Moderate,Consider developing Speaking as a medium-prior...,Office Manager,55.624220,Moderate recommendation based on combined cand...
5,Candidate_002,Office Manager,Active Listening,4.00,3.88,0.0,0.788,52.86,0.684,68.42,...,13,True,CIP pathway available,Review mapped CIP education pathways,Medium,Moderate,Consider developing Active Listening as a medi...,Office Manager,55.624220,Moderate recommendation based on combined cand...
6,Candidate_003,Administrative Assistant,Mathematics,2.00,1.62,0.0,0.362,43.85,0.393,39.26,...,2,True,CIP pathway available,Review mapped CIP education pathways,Lower,Moderate,Consider developing Mathematics as an addition...,Administrative Assistant,93.244151,Strong match based on both baseline candidate ...
7,Candidate_003,Administrative Assistant,Science,1.00,0.00,0.0,0.100,43.85,0.235,23.54,...,2,True,CIP pathway available,Review mapped CIP education pathways,Lower,Moderate,Consider developing Science as an additional s...,Administrative Assistant,93.244151,Strong match based on both baseline candidate ...
8,Candidate_004,Software Developer,Active Listening,3.38,3.88,0.0,0.726,22.91,0.527,52.72,...,20,True,CIP pathway available,Review mapped CIP education pathways,Medium,Lower,Consider developing Active Listening as a medi...,Software Developer,78.591423,Strong match based on both baseline candidate ...
9,Candidate_004,Software Developer,Speaking,3.12,3.62,0.0,0.674,22.91,0.496,49.60,...,20,True,CIP pathway available,Review mapped CIP education pathways,Lower,Lower,Consider developing Speaking as an additional ...,Software Developer,78.591423,Strong match based on both baseline candidate ...


In [17]:
# ============================================================
# WEEK 8 — CELL 6B
# INSPECT MISSING SKILL PRIORITY DATA FOR CORRECT TOP-1 MATCHING
# ============================================================

print("=" * 70)
print("WEEK 8 — MISSING SKILL PRIORITY INSPECTION")
print("=" * 70)

print("\nShape:")
print(missing_skill_priorities.shape)

print("\nColumns:")
print(missing_skill_priorities.columns.tolist())

print("\nUnique candidates:")
print(missing_skill_priorities["candidate_id"].nunique())

print("\nUnique occupations:")
print(missing_skill_priorities["selected_occupation"].nunique())

print("\nSample:")
display(missing_skill_priorities.head(10))

WEEK 8 — MISSING SKILL PRIORITY INSPECTION

Shape:
(2610, 19)

Columns:
['candidate_id', 'selected_occupation', 'onet_soc_code', 'onet_title', 'skill', 'importance', 'level', 'candidate_evidence', 'skill_gap_priority', 'skill_priority_rank', 'posting_count', 'total_vacancies', 'province_count', 'demand_score', 'demand_percentage', 'demand_normalized', 'prescriptive_priority_score', 'prescriptive_priority_percentage', 'prescriptive_rank']

Unique candidates:
56

Unique occupations:
15

Sample:


,candidate_id,selected_occupation,onet_soc_code,onet_title,skill,importance,level,candidate_evidence,skill_gap_priority,skill_priority_rank,posting_count,total_vacancies,province_count,demand_score,demand_percentage,demand_normalized,prescriptive_priority_score,prescriptive_priority_percentage,prescriptive_rank
0,Candidate_001,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Exc...",Active Listening,4.00,3.75,0.0,0.775,1,812,890,11,0.438451,43.85,0.4385,0.640,64.04,1
1,Candidate_001,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Exc...",Speaking,4.00,3.62,0.0,0.762,2,812,890,11,0.438451,43.85,0.4385,0.633,63.26,2
2,Candidate_001,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Exc...",Monitoring,3.12,3.25,0.0,0.637,3,812,890,11,0.438451,43.85,0.4385,0.558,55.76,3
3,Candidate_001,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",Active Listening,3.25,3.75,0.0,0.700,1,396,422,11,0.299226,29.92,0.2992,0.540,53.97,1
4,Candidate_001,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",Speaking,3.12,3.12,0.0,0.624,2,396,422,11,0.299226,29.92,0.2992,0.494,49.41,2
5,Candidate_001,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",Monitoring,3.00,2.88,0.0,0.588,3,396,422,11,0.299226,29.92,0.2992,0.472,47.25,3
6,Candidate_001,Continuing Care Assistant,31-1131.00,Nursing Assistants,Active Listening,3.62,3.00,0.0,0.662,1,819,1172,9,0.439620,43.96,0.4396,0.573,57.30,1
7,Candidate_001,Continuing Care Assistant,31-1131.00,Nursing Assistants,Monitoring,3.25,3.00,0.0,0.625,2,819,1172,9,0.439620,43.96,0.4396,0.551,55.08,2
8,Candidate_001,Continuing Care Assistant,31-1131.00,Nursing Assistants,Speaking,3.12,3.00,0.0,0.612,3,819,1172,9,0.439620,43.96,0.4396,0.543,54.30,3
9,Candidate_001,Delivery Driver,53-3033.00,Light Truck Drivers,Active Listening,3.12,2.88,0.0,0.600,1,557,622,12,0.370531,37.05,0.3705,0.508,50.82,1


In [19]:
# ============================================================
# WEEK 8 — CELL 6C
# BUILD CORRECT TOP-1 SKILL-GAP SUMMARY
# ============================================================

# ------------------------------------------------------------
# Prepare actual Top-1 candidate + occupation pairs
# ------------------------------------------------------------

top1_pairs = top1_candidate_summary[
    ["candidate_id", "top_recommended_occupation"]
].rename(
    columns={"top_recommended_occupation": "selected_occupation"}
)

# ------------------------------------------------------------
# Keep missing skills only for the ACTUAL Top-1 occupation
# ------------------------------------------------------------

top1_missing_skills = top1_pairs.merge(
    missing_skill_priorities,
    on=["candidate_id", "selected_occupation"],
    how="left"
)

# Keep only rows where a missing skill exists
top1_missing_skills_valid = top1_missing_skills[
    top1_missing_skills["skill"].notna()
].copy()

# ------------------------------------------------------------
# Sort skills by priority
# ------------------------------------------------------------

top1_missing_skills_valid = top1_missing_skills_valid.sort_values(
    [
        "candidate_id",
        "prescriptive_priority_percentage"
    ],
    ascending=[True, False]
)

# ------------------------------------------------------------
# Build candidate-level Top-1 skill-gap summary
# ------------------------------------------------------------

def build_skill_gap_summary(group):

    group = group.sort_values(
        "prescriptive_priority_percentage",
        ascending=False
    )

    skill_list = [
        f"{row.skill} ({row.prescriptive_priority_percentage:.2f}%)"
        for row in group.itertuples()
    ]

    top3_skills = skill_list[:3]

    return pd.Series({
        "missing_skill_count": len(group),

        "prioritized_skill_gaps": "; ".join(top3_skills),

        "highest_priority_skill": group.iloc[0]["skill"],

        "highest_skill_priority_percentage":
            round(group.iloc[0]["prescriptive_priority_percentage"], 2),

        "average_skill_priority_percentage":
            round(group["prescriptive_priority_percentage"].mean(), 2),

        "skill_development_action":
            "Develop priority skills: " +
            ", ".join(group.head(3)["skill"].tolist())
    })


top1_skill_gap_summary = (
    top1_missing_skills_valid
    .groupby(
        ["candidate_id", "selected_occupation"],
        as_index=False
    )
    .apply(build_skill_gap_summary, include_groups=False)
    .reset_index()
)

# Remove possible extra index column created by groupby/apply
if "level_2" in top1_skill_gap_summary.columns:
    top1_skill_gap_summary = top1_skill_gap_summary.drop(columns=["level_2"])

# ------------------------------------------------------------
# Validate against actual Top-1 recommendations
# ------------------------------------------------------------

validation = top1_skill_gap_summary.merge(
    top1_candidate_summary[
        ["candidate_id", "top_recommended_occupation"]
    ],
    on="candidate_id",
    how="left"
)

validation["occupation_match"] = (
    validation["selected_occupation"]
    ==
    validation["top_recommended_occupation"]
)

print("=" * 70)
print("WEEK 8 — CORRECT TOP-1 SKILL-GAP SUMMARY")
print("=" * 70)

print("\nCandidates with Top-1 missing skills:")
print(top1_skill_gap_summary["candidate_id"].nunique())

print("\nCandidates with no Top-1 missing skills:")
print(
    63 - top1_skill_gap_summary["candidate_id"].nunique()
)

print("\nOccupation matches:")
print(validation["occupation_match"].value_counts())

print("\nSample:")
display(top1_skill_gap_summary.head(10))

assert validation["occupation_match"].all()

print(
    "\n✅ Top-1 skill gaps now correctly match "
    "the final hybrid recommendation."
)

WEEK 8 — CORRECT TOP-1 SKILL-GAP SUMMARY

Candidates with Top-1 missing skills:
56

Candidates with no Top-1 missing skills:
7

Occupation matches:
occupation_match
True    56
Name: count, dtype: int64

Sample:


,index,candidate_id,selected_occupation,missing_skill_count,prioritized_skill_gaps,highest_priority_skill,highest_skill_priority_percentage,average_skill_priority_percentage,skill_development_action
0,0,Candidate_001,Software Developer,3,Active Listening (52.72%); Speaking (49.60%); ...,Active Listening,52.72,50.16,"Develop priority skills: Active Listening, Spe..."
1,1,Candidate_002,Software Developer,5,Active Listening (52.72%); Speaking (49.60%); ...,Active Listening,52.72,45.76,"Develop priority skills: Active Listening, Spe..."
2,2,Candidate_003,Administrative Assistant,2,Mathematics (39.26%); Science (23.54%),Mathematics,39.26,31.40,"Develop priority skills: Mathematics, Science"
3,3,Candidate_004,Software Developer,2,Active Listening (52.72%); Speaking (49.60%),Active Listening,52.72,51.16,"Develop priority skills: Active Listening, Spe..."
4,4,Candidate_005,Software Developer,5,Active Listening (52.72%); Writing (50.38%); S...,Active Listening,52.72,46.80,"Develop priority skills: Active Listening, Wri..."
5,5,Candidate_006,Software Developer,1,Monitoring (48.16%),Monitoring,48.16,48.16,Develop priority skills: Monitoring
6,6,Candidate_008,Software Developer,3,Active Listening (52.72%); Writing (50.38%); S...,Active Listening,52.72,50.90,"Develop priority skills: Active Listening, Wri..."
7,7,Candidate_009,Administrative Assistant,8,Active Listening (64.04%); Speaking (63.26%); ...,Active Listening,64.04,49.68,"Develop priority skills: Active Listening, Spe..."
8,8,Candidate_010,Food Service Supervisor,1,Science (33.33%),Science,33.33,33.33,Develop priority skills: Science
9,9,Candidate_011,Office Manager,4,Speaking (69.14%); Active Listening (68.42%); ...,Speaking,69.14,54.71,"Develop priority skills: Speaking, Active List..."



✅ Top-1 skill gaps now correctly match the final hybrid recommendation.


In [21]:
# ============================================================
# WEEK 8 — CELL 7
# BUILD DEPLOYMENT-READY CANDIDATE PROFILE
# ============================================================

# ------------------------------------------------------------
# Merge Top-1 recommendation with corrected skill-gap summary
# ------------------------------------------------------------

candidate_deployment_profile = top1_candidate_summary.merge(
    top1_skill_gap_summary,
    left_on=[
        "candidate_id",
        "top_recommended_occupation"
    ],
    right_on=[
        "candidate_id",
        "selected_occupation"
    ],
    how="left"
)

# Remove duplicate occupation field from skill-gap table
candidate_deployment_profile = candidate_deployment_profile.drop(
    columns=["selected_occupation"]
)

# ------------------------------------------------------------
# Handle candidates with no missing skills
# ------------------------------------------------------------

candidate_deployment_profile["missing_skill_count"] = (
    candidate_deployment_profile["missing_skill_count"]
    .fillna(0)
    .astype(int)
)

candidate_deployment_profile["prioritized_skill_gaps"] = (
    candidate_deployment_profile["prioritized_skill_gaps"]
    .fillna("No priority skill gaps identified")
)

candidate_deployment_profile["highest_priority_skill"] = (
    candidate_deployment_profile["highest_priority_skill"]
    .fillna("None")
)

candidate_deployment_profile["skill_development_action"] = (
    candidate_deployment_profile["skill_development_action"]
    .fillna(
        "Maintain and strengthen existing skills for the recommended occupation"
    )
)

# ------------------------------------------------------------
# Add skill-gap status
# ------------------------------------------------------------

candidate_deployment_profile["skill_gap_status"] = np.where(
    candidate_deployment_profile["missing_skill_count"] > 0,
    "Skill development recommended",
    "No priority skill gaps identified"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — DEPLOYMENT CANDIDATE PROFILE")
print("=" * 70)

print("\nShape:")
print(candidate_deployment_profile.shape)

print("\nUnique candidates:")
print(candidate_deployment_profile["candidate_id"].nunique())

print("\nSkill-gap status:")
display(
    candidate_deployment_profile[
        "skill_gap_status"
    ]
    .value_counts()
    .rename_axis("skill_gap_status")
    .reset_index(name="candidate_count")
)

print("\nMissing-skill count summary:")
display(
    candidate_deployment_profile[
        "missing_skill_count"
    ].describe().to_frame()
)

print("\nSample:")
display(candidate_deployment_profile.head(10))

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(candidate_deployment_profile) == 63
assert candidate_deployment_profile["candidate_id"].nunique() == 63
assert candidate_deployment_profile["candidate_id"].duplicated().sum() == 0
assert candidate_deployment_profile["top_recommended_occupation"].notna().all()

print(
    "\n✅ Deployment-ready candidate recommendation and "
    "skill-gap profile created successfully."
)

WEEK 8 — DEPLOYMENT CANDIDATE PROFILE

Shape:
(63, 16)

Unique candidates:
63

Skill-gap status:


,skill_gap_status,candidate_count
0,Skill development recommended,56
1,No priority skill gaps identified,7



Missing-skill count summary:


,missing_skill_count
count,63.000000
mean,2.761905
std,1.729388
min,0.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,8.000000



Sample:


,candidate_id,top_recommended_occupation,top_recommendation_score,top_recommendation_percentage,top_skill_percentage,top_semantic_percentage,top_demand_percentage,top_education_percentage,index,missing_skill_count,prioritized_skill_gaps,highest_priority_skill,highest_skill_priority_percentage,average_skill_priority_percentage,skill_development_action,skill_gap_status
0,Candidate_001,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,0.0,3,Active Listening (52.72%); Speaking (49.60%); ...,Active Listening,52.72,50.16,"Develop priority skills: Active Listening, Spe...",Skill development recommended
1,Candidate_002,Software Developer,0.782459,78.25,81.90,100.00,22.91,100.0,1.0,5,Active Listening (52.72%); Speaking (49.60%); ...,Active Listening,52.72,45.76,"Develop priority skills: Active Listening, Spe...",Skill development recommended
2,Candidate_003,Administrative Assistant,0.743337,74.33,100.00,58.76,43.85,100.0,2.0,2,Mathematics (39.26%); Science (23.54%),Mathematics,39.26,31.40,"Develop priority skills: Mathematics, Science",Skill development recommended
3,Candidate_004,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,3.0,2,Active Listening (52.72%); Speaking (49.60%),Active Listening,52.72,51.16,"Develop priority skills: Active Listening, Spe...",Skill development recommended
4,Candidate_005,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,4.0,5,Active Listening (52.72%); Writing (50.38%); S...,Active Listening,52.72,46.80,"Develop priority skills: Active Listening, Wri...",Skill development recommended
5,Candidate_006,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,5.0,1,Monitoring (48.16%),Monitoring,48.16,48.16,Develop priority skills: Monitoring,Skill development recommended
6,Candidate_007,Food Service Supervisor,0.743113,74.31,100.00,44.70,68.33,100.0,NaN,0,No priority skill gaps identified,None,NaN,NaN,Maintain and strengthen existing skills for th...,No priority skill gaps identified
7,Candidate_008,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,6.0,3,Active Listening (52.72%); Writing (50.38%); S...,Active Listening,52.72,50.90,"Develop priority skills: Active Listening, Wri...",Skill development recommended
8,Candidate_009,Administrative Assistant,0.741840,74.18,100.00,58.33,43.85,100.0,7.0,8,Active Listening (64.04%); Speaking (63.26%); ...,Active Listening,64.04,49.68,"Develop priority skills: Active Listening, Spe...",Skill development recommended
9,Candidate_010,Food Service Supervisor,0.721099,72.11,92.66,45.75,68.33,100.0,8.0,1,Science (33.33%),Science,33.33,33.33,Develop priority skills: Science,Skill development recommended



✅ Deployment-ready candidate recommendation and skill-gap profile created successfully.


In [23]:
# ============================================================
# WEEK 8 — CELL 8
# INSPECT CANDIDATE RAG EXPLANATION DATA
# ============================================================

print("=" * 70)
print("WEEK 8 — RAG EXPLANATION INSPECTION")
print("=" * 70)

print("\nShape:")
print(candidate_rag_explanations.shape)

print("\nColumns:")
print(candidate_rag_explanations.columns.tolist())

print("\nUnique candidates:")
print(candidate_rag_explanations["candidate_id"].nunique())

print("\nDuplicate candidate IDs:")
print(candidate_rag_explanations["candidate_id"].duplicated().sum())

print("\nSample:")
display(candidate_rag_explanations.head(5))

WEEK 8 — RAG EXPLANATION INSPECTION

Shape:
(63, 14)

Columns:
['candidate_id', 'recommended_occupation', 'hybrid_recommendation_score', 'missing_skill_count', 'priority_missing_skills', 'demand_percentage', 'demand_level', 'cip_status', 'education_action', 'skill_source_ids', 'task_source_ids', 'demand_source_ids', 'education_source_ids', 'rag_explanation']

Unique candidates:
63

Duplicate candidate IDs:
0

Sample:


,candidate_id,recommended_occupation,hybrid_recommendation_score,missing_skill_count,priority_missing_skills,demand_percentage,demand_level,cip_status,education_action,skill_source_ids,task_source_ids,demand_source_ids,education_source_ids,rag_explanation
0,Candidate_001,Software Developer,69.0545,3,"Active Listening, Speaking, Monitoring",22.91,Lower,CIP pathway available,Review mapped CIP education pathways,RAG_DOC_0162; RAG_DOC_0169; RAG_DOC_0166,RAG_DOC_0256; RAG_DOC_0253; RAG_DOC_0250,RAG_DOC_0672,RAG_DOC_0673; RAG_DOC_0707,Recommendation: Software Developer is the Top-...
1,Candidate_002,Office Manager,55.6242,3,"Monitoring, Speaking, Active Listening",52.86,Moderate,CIP pathway available,Review mapped CIP education pathways,RAG_DOC_0126; RAG_DOC_0129; RAG_DOC_0122,RAG_DOC_0518; RAG_DOC_0480; RAG_DOC_0508,RAG_DOC_0664,RAG_DOC_0678; RAG_DOC_0703,Recommendation: Office Manager is the Top-1 ca...
2,Candidate_003,Administrative Assistant,93.2442,2,"Mathematics, Science",43.85,Moderate,CIP pathway available,Review mapped CIP education pathways,RAG_DOC_0025; RAG_DOC_0028,RAG_DOC_0605; RAG_DOC_0590; RAG_DOC_0606,RAG_DOC_0666,RAG_DOC_0680; RAG_DOC_0688,Recommendation: Administrative Assistant is th...
3,Candidate_004,Software Developer,78.5914,2,"Active Listening, Speaking",22.91,Lower,CIP pathway available,Review mapped CIP education pathways,RAG_DOC_0162; RAG_DOC_0169,RAG_DOC_0256; RAG_DOC_0253; RAG_DOC_0250,RAG_DOC_0672,RAG_DOC_0673; RAG_DOC_0707,Recommendation: Software Developer is the Top-...
4,Candidate_005,Software Developer,53.3510,3,"Active Listening, Writing, Speaking",22.91,Lower,CIP pathway available,Review mapped CIP education pathways,RAG_DOC_0162; RAG_DOC_0170; RAG_DOC_0169,RAG_DOC_0256; RAG_DOC_0253; RAG_DOC_0250,RAG_DOC_0672,RAG_DOC_0673; RAG_DOC_0707,Recommendation: Software Developer is the Top-...


In [25]:
# ============================================================
# WEEK 8 — CELL 8A
# VALIDATE RAG OCCUPATION AGAINST FINAL HYBRID TOP-1
# ============================================================

rag_alignment_check = (
    top1_candidate_summary[
        ["candidate_id", "top_recommended_occupation"]
    ]
    .merge(
        candidate_rag_explanations[
            [
                "candidate_id",
                "recommended_occupation",
                "rag_explanation"
            ]
        ],
        on="candidate_id",
        how="left"
    )
)

# ------------------------------------------------------------
# Check occupation alignment
# ------------------------------------------------------------

rag_alignment_check["rag_occupation_match"] = (
    rag_alignment_check["top_recommended_occupation"]
    ==
    rag_alignment_check["recommended_occupation"]
)

print("=" * 70)
print("WEEK 8 — RAG OCCUPATION ALIGNMENT CHECK")
print("=" * 70)

print("\nTotal candidates:")
print(len(rag_alignment_check))

print("\nRAG occupation alignment:")
display(
    rag_alignment_check[
        "rag_occupation_match"
    ]
    .value_counts(dropna=False)
    .rename_axis("occupation_match")
    .reset_index(name="candidate_count")
)

print("\nCandidates with mismatched RAG occupation:")
display(
    rag_alignment_check.loc[
        ~rag_alignment_check["rag_occupation_match"],
        [
            "candidate_id",
            "top_recommended_occupation",
            "recommended_occupation"
        ]
    ]
)

print("\nMatching percentage:")
print(
    round(
        rag_alignment_check["rag_occupation_match"].mean() * 100,
        2
    ),
    "%"
)

WEEK 8 — RAG OCCUPATION ALIGNMENT CHECK

Total candidates:
63

RAG occupation alignment:


,occupation_match,candidate_count
0,True,47
1,False,16



Candidates with mismatched RAG occupation:


,candidate_id,top_recommended_occupation,recommended_occupation
1,Candidate_002,Software Developer,Office Manager
9,Candidate_010,Food Service Supervisor,Office Manager
21,Candidate_022,Food Service Supervisor,Office Manager
22,Candidate_023,Software Developer,Office Manager
23,Candidate_024,Software Developer,Office Manager
26,Candidate_027,Software Developer,Office Manager
27,Candidate_028,Software Developer,Office Manager
28,Candidate_029,Administrative Assistant,Food Service Supervisor
29,Candidate_030,Software Developer,Office Manager
33,Candidate_034,Food Service Supervisor,Office Manager



Matching percentage:
74.6 %


In [27]:
# ============================================================
# WEEK 8 — CELL 8B
# BUILD SAFE RAG DEPLOYMENT TABLE
# KEEP ONLY OCCUPATION-ALIGNED EXPLANATIONS
# ============================================================

# ------------------------------------------------------------
# Start from final hybrid Top-1 recommendation
# ------------------------------------------------------------

deployment_rag = (
    top1_candidate_summary[
        ["candidate_id", "top_recommended_occupation"]
    ]
    .merge(
        candidate_rag_explanations,
        on="candidate_id",
        how="left",
        suffixes=("", "_rag")
    )
)

# ------------------------------------------------------------
# Check whether Week 7 RAG explanation matches
# final hybrid Top-1 occupation
# ------------------------------------------------------------

deployment_rag["rag_occupation_match"] = (
    deployment_rag["top_recommended_occupation"]
    ==
    deployment_rag["recommended_occupation"]
)

# ------------------------------------------------------------
# Keep explanation ONLY when occupation matches
# ------------------------------------------------------------

deployment_rag["deployment_rag_explanation"] = np.where(
    deployment_rag["rag_occupation_match"],
    deployment_rag["rag_explanation"],
    np.nan
)

deployment_rag["rag_deployment_status"] = np.where(
    deployment_rag["rag_occupation_match"],
    "Ready",
    "Regeneration required"
)

# ------------------------------------------------------------
# Create clean deployment RAG table
# ------------------------------------------------------------

deployment_rag_safe = deployment_rag[
    [
        "candidate_id",
        "top_recommended_occupation",
        "rag_occupation_match",
        "rag_deployment_status",
        "deployment_rag_explanation"
    ]
].copy()

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — SAFE RAG DEPLOYMENT TABLE")
print("=" * 70)

print("\nShape:")
print(deployment_rag_safe.shape)

print("\nUnique candidates:")
print(deployment_rag_safe["candidate_id"].nunique())

print("\nRAG deployment status:")
display(
    deployment_rag_safe[
        "rag_deployment_status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="candidate_count")
)

print("\nValid deployment explanations:")
print(
    deployment_rag_safe[
        "deployment_rag_explanation"
    ].notna().sum()
)

print("\nExplanations requiring regeneration:")
print(
    deployment_rag_safe[
        "deployment_rag_explanation"
    ].isna().sum()
)

print("\nMismatched candidates:")
display(
    deployment_rag_safe[
        deployment_rag_safe["rag_deployment_status"]
        == "Regeneration required"
    ][
        [
            "candidate_id",
            "top_recommended_occupation",
            "rag_deployment_status"
        ]
    ]
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(deployment_rag_safe) == 63
assert deployment_rag_safe["candidate_id"].nunique() == 63

assert (
    deployment_rag_safe.loc[
        ~deployment_rag_safe["rag_occupation_match"],
        "deployment_rag_explanation"
    ]
    .isna()
    .all()
)

print(
    "\n✅ Only occupation-aligned RAG explanations "
    "are currently approved for deployment."
)

WEEK 8 — SAFE RAG DEPLOYMENT TABLE

Shape:
(63, 5)

Unique candidates:
63

RAG deployment status:


,status,candidate_count
0,Ready,47
1,Regeneration required,16



Valid deployment explanations:
47

Explanations requiring regeneration:
16

Mismatched candidates:


,candidate_id,top_recommended_occupation,rag_deployment_status
1,Candidate_002,Software Developer,Regeneration required
9,Candidate_010,Food Service Supervisor,Regeneration required
21,Candidate_022,Food Service Supervisor,Regeneration required
22,Candidate_023,Software Developer,Regeneration required
23,Candidate_024,Software Developer,Regeneration required
26,Candidate_027,Software Developer,Regeneration required
27,Candidate_028,Software Developer,Regeneration required
28,Candidate_029,Administrative Assistant,Regeneration required
29,Candidate_030,Software Developer,Regeneration required
33,Candidate_034,Food Service Supervisor,Regeneration required



✅ Only occupation-aligned RAG explanations are currently approved for deployment.


In [29]:
# ============================================================
# WEEK 8 — CELL 8C
# INSPECT RAG KNOWLEDGE BASE FOR REGENERATION
# ============================================================

print("=" * 70)
print("WEEK 8 — RAG KNOWLEDGE BASE INSPECTION")
print("=" * 70)

print("\nShape:")
print(rag_knowledge_base.shape)

print("\nColumns:")
print(rag_knowledge_base.columns.tolist())

print("\nSource type distribution:")
display(
    rag_knowledge_base[
        "source_type"
    ]
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="document_count")
)

print("\nDocuments by occupation:")
display(
    rag_knowledge_base[
        "selected_occupation"
    ]
    .value_counts(dropna=False)
    .rename_axis("occupation")
    .reset_index(name="document_count")
)

# ------------------------------------------------------------
# Check knowledge-base coverage for the 16 occupations
# requiring regenerated explanations
# ------------------------------------------------------------

rag_regeneration_targets = (
    deployment_rag_safe[
        deployment_rag_safe["rag_deployment_status"]
        == "Regeneration required"
    ][
        ["candidate_id", "top_recommended_occupation"]
    ]
    .copy()
)

target_occupations = sorted(
    rag_regeneration_targets[
        "top_recommended_occupation"
    ].unique()
)

print("\nOccupations requiring RAG regeneration:")
print(target_occupations)

coverage_rows = []

for occupation in target_occupations:

    occupation_docs = rag_knowledge_base[
        rag_knowledge_base["selected_occupation"]
        == occupation
    ]

    coverage_rows.append({
        "occupation": occupation,
        "document_count": len(occupation_docs),
        "source_types": ", ".join(
            sorted(
                occupation_docs[
                    "source_type"
                ].dropna().unique()
            )
        )
    })

rag_regeneration_coverage = pd.DataFrame(coverage_rows)

print("\nRAG coverage for regeneration targets:")
display(rag_regeneration_coverage)

print("\nSample knowledge-base documents:")
display(
    rag_knowledge_base[
        rag_knowledge_base["selected_occupation"].isin(
            target_occupations
        )
    ].head(10)
)

WEEK 8 — RAG KNOWLEDGE BASE INSPECTION

Shape:
(707, 6)

Columns:
['document_id', 'selected_occupation', 'source_type', 'source_name', 'source_reference', 'document_text']

Source type distribution:


,source_type,document_count
0,O*NET Task,487
1,O*NET Skill,150
2,Occupation Profile,20
3,Integrated Occupation Profile,20
4,Canadian Labour Demand,15
5,CIP Education Pathway,15



Documents by occupation:


,occupation,document_count
0,Information Technology (IT) Analyst,100
1,Continuing Care Assistant,71
2,Office Administrator,66
3,Secondary School Teacher,46
4,Administrative Assistant,45
5,"Driver, Truck",43
6,Bookkeeper,42
7,Office Manager,42
8,Restaurant Manager,42
9,Food Service Supervisor,40



Occupations requiring RAG regeneration:
['Administrative Assistant', 'Food Service Supervisor', 'Software Developer']

RAG coverage for regeneration targets:


,occupation,document_count,source_types
0,Administrative Assistant,45,"CIP Education Pathway, Canadian Labour Demand,..."
1,Food Service Supervisor,40,"CIP Education Pathway, Canadian Labour Demand,..."
2,Software Developer,31,"CIP Education Pathway, Canadian Labour Demand,..."



Sample knowledge-base documents:


,document_id,selected_occupation,source_type,source_name,source_reference,document_text
0,RAG_DOC_0001,Software Developer,Occupation Profile,O*NET,15-1252.00,Occupation: Software Developer. O*NET occupati...
5,RAG_DOC_0006,Administrative Assistant,Occupation Profile,O*NET,43-6014.00,Occupation: Administrative Assistant. O*NET oc...
11,RAG_DOC_0012,Food Service Supervisor,Occupation Profile,O*NET,35-1012.00,Occupation: Food Service Supervisor. O*NET occ...
20,RAG_DOC_0021,Administrative Assistant,O*NET Skill,O*NET,43-6014.00,"For Administrative Assistant, O*NET identifies..."
21,RAG_DOC_0022,Administrative Assistant,O*NET Skill,O*NET,43-6014.00,"For Administrative Assistant, O*NET identifies..."
22,RAG_DOC_0023,Administrative Assistant,O*NET Skill,O*NET,43-6014.00,"For Administrative Assistant, O*NET identifies..."
23,RAG_DOC_0024,Administrative Assistant,O*NET Skill,O*NET,43-6014.00,"For Administrative Assistant, O*NET identifies..."
24,RAG_DOC_0025,Administrative Assistant,O*NET Skill,O*NET,43-6014.00,"For Administrative Assistant, O*NET identifies..."
25,RAG_DOC_0026,Administrative Assistant,O*NET Skill,O*NET,43-6014.00,"For Administrative Assistant, O*NET identifies..."
26,RAG_DOC_0027,Administrative Assistant,O*NET Skill,O*NET,43-6014.00,"For Administrative Assistant, O*NET identifies..."


In [31]:
# ============================================================
# WEEK 8 — CELL 8D
# BUILD GROUNDED RAG SOURCE PACKAGES FOR REGENERATION
# ============================================================

def get_rag_source_package(candidate_id, occupation):
    
    # --------------------------------------------------------
    # Candidate's actual Top-1 missing skills
    # --------------------------------------------------------
    
    candidate_skills = top1_missing_skills_valid[
        (top1_missing_skills_valid["candidate_id"] == candidate_id) &
        (top1_missing_skills_valid["selected_occupation"] == occupation)
    ].sort_values(
        "prescriptive_priority_percentage",
        ascending=False
    )

    # --------------------------------------------------------
    # Knowledge-base documents for the occupation
    # --------------------------------------------------------
    
    occupation_docs = rag_knowledge_base[
        rag_knowledge_base["selected_occupation"] == occupation
    ].copy()

    # --------------------------------------------------------
    # Select grounded evidence by source type
    # --------------------------------------------------------
    
    profile_docs = occupation_docs[
        occupation_docs["source_type"].isin([
            "Occupation Profile",
            "Integrated Occupation Profile"
        ])
    ].head(2)

    skill_docs = occupation_docs[
        occupation_docs["source_type"] == "O*NET Skill"
    ]

    task_docs = occupation_docs[
        occupation_docs["source_type"] == "O*NET Task"
    ].head(3)

    demand_docs = occupation_docs[
        occupation_docs["source_type"] == "Canadian Labour Demand"
    ].head(1)

    education_docs = occupation_docs[
        occupation_docs["source_type"] == "CIP Education Pathway"
    ].head(1)

    # --------------------------------------------------------
    # Prioritize skill evidence corresponding to candidate gaps
    # --------------------------------------------------------
    
    missing_skills = candidate_skills[
        "skill"
    ].head(3).tolist()

    selected_skill_docs = []

    for skill in missing_skills:
        matches = skill_docs[
            skill_docs["document_text"]
            .str.contains(
                str(skill),
                case=False,
                na=False,
                regex=False
            )
        ]

        if len(matches) > 0:
            selected_skill_docs.append(matches.iloc[0])

    if selected_skill_docs:
        selected_skill_docs = pd.DataFrame(selected_skill_docs)
    else:
        selected_skill_docs = skill_docs.head(3)

    # --------------------------------------------------------
    # Combine evidence
    # --------------------------------------------------------
    
    evidence = pd.concat(
        [
            profile_docs,
            selected_skill_docs,
            task_docs,
            demand_docs,
            education_docs
        ],
        ignore_index=True
    ).drop_duplicates(
        subset=["document_id"]
    )

    return evidence


# ------------------------------------------------------------
# Build source packages for all 16 regeneration candidates
# ------------------------------------------------------------

source_package_rows = []

for row in rag_regeneration_targets.itertuples(index=False):

    evidence = get_rag_source_package(
        row.candidate_id,
        row.top_recommended_occupation
    )

    source_package_rows.append({
        "candidate_id": row.candidate_id,
        "recommended_occupation":
            row.top_recommended_occupation,

        "evidence_document_count":
            len(evidence),

        "evidence_source_types":
            "; ".join(
                sorted(
                    evidence[
                        "source_type"
                    ].dropna().unique()
                )
            ),

        "evidence_document_ids":
            "; ".join(
                evidence[
                    "document_id"
                ].astype(str)
            )
    })


rag_regeneration_source_packages = pd.DataFrame(
    source_package_rows
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — RAG REGENERATION SOURCE PACKAGES")
print("=" * 70)

print("\nShape:")
print(rag_regeneration_source_packages.shape)

print("\nUnique candidates:")
print(
    rag_regeneration_source_packages[
        "candidate_id"
    ].nunique()
)

print("\nEvidence document count summary:")
display(
    rag_regeneration_source_packages[
        "evidence_document_count"
    ].describe().to_frame()
)

print("\nSource packages:")
display(rag_regeneration_source_packages)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(rag_regeneration_source_packages) == 16

assert (
    rag_regeneration_source_packages[
        "candidate_id"
    ].nunique()
    == 16
)

assert (
    rag_regeneration_source_packages[
        "evidence_document_count"
    ] > 0
).all()

print(
    "\n✅ Grounded evidence packages created for all "
    "16 RAG explanations requiring regeneration."
)

WEEK 8 — RAG REGENERATION SOURCE PACKAGES

Shape:
(16, 5)

Unique candidates:
16

Evidence document count summary:


,evidence_document_count
count,16.000000
mean,9.375000
std,0.957427
min,8.000000
25%,8.000000
50%,10.000000
75%,10.000000
max,10.000000



Source packages:


,candidate_id,recommended_occupation,evidence_document_count,evidence_source_types,evidence_document_ids
0,Candidate_002,Software Developer,10,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0001; RAG_DOC_0707; RAG_DOC_0162; RAG_...
1,Candidate_010,Food Service Supervisor,8,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0012; RAG_DOC_0694; RAG_DOC_0078; RAG_...
2,Candidate_022,Food Service Supervisor,8,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0012; RAG_DOC_0694; RAG_DOC_0078; RAG_...
3,Candidate_023,Software Developer,10,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0001; RAG_DOC_0707; RAG_DOC_0162; RAG_...
4,Candidate_024,Software Developer,10,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0001; RAG_DOC_0707; RAG_DOC_0162; RAG_...
5,Candidate_027,Software Developer,10,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0001; RAG_DOC_0707; RAG_DOC_0162; RAG_...
6,Candidate_028,Software Developer,10,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0001; RAG_DOC_0707; RAG_DOC_0162; RAG_...
7,Candidate_029,Administrative Assistant,10,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0006; RAG_DOC_0688; RAG_DOC_0021; RAG_...
8,Candidate_030,Software Developer,10,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0001; RAG_DOC_0707; RAG_DOC_0162; RAG_...
9,Candidate_034,Food Service Supervisor,8,CIP Education Pathway; Canadian Labour Demand;...,RAG_DOC_0012; RAG_DOC_0694; RAG_DOC_0078; RAG_...



✅ Grounded evidence packages created for all 16 RAG explanations requiring regeneration.


In [33]:
# ============================================================
# WEEK 8 — CELL 8E
# REGENERATE GROUNDED RAG EXPLANATIONS FOR THE 16 MISMATCHES
# ============================================================

def regenerate_grounded_explanation(candidate_id, occupation):

    # --------------------------------------------------------
    # Candidate Top-1 recommendation information
    # --------------------------------------------------------

    candidate_row = top1_candidate_summary[
        top1_candidate_summary["candidate_id"] == candidate_id
    ].iloc[0]

    recommendation_percentage = round(
        candidate_row["top_recommendation_percentage"], 2
    )

    skill_percentage = round(
        candidate_row["top_skill_percentage"], 2
    )

    semantic_percentage = round(
        candidate_row["top_semantic_percentage"], 2
    )

    demand_percentage = round(
        candidate_row["top_demand_percentage"], 2
    )

    education_percentage = round(
        candidate_row["top_education_percentage"], 2
    )

    # --------------------------------------------------------
    # Candidate skill gaps for the ACTUAL Top-1 occupation
    # --------------------------------------------------------

    skill_gap_row = top1_skill_gap_summary[
        (top1_skill_gap_summary["candidate_id"] == candidate_id) &
        (top1_skill_gap_summary["selected_occupation"] == occupation)
    ]

    if len(skill_gap_row) > 0:

        skill_gap_row = skill_gap_row.iloc[0]

        missing_skill_count = int(
            skill_gap_row["missing_skill_count"]
        )

        prioritized_skills = (
            skill_gap_row["prioritized_skill_gaps"]
        )

        development_action = (
            skill_gap_row["skill_development_action"]
        )

    else:

        missing_skill_count = 0

        prioritized_skills = (
            "No priority skill gaps identified"
        )

        development_action = (
            "Maintain and strengthen existing skills."
        )

    # --------------------------------------------------------
    # RAG evidence package
    # --------------------------------------------------------

    source_row = rag_regeneration_source_packages[
        rag_regeneration_source_packages["candidate_id"]
        == candidate_id
    ].iloc[0]

    source_ids = source_row["evidence_document_ids"]

    source_types = source_row["evidence_source_types"]

    # --------------------------------------------------------
    # Create grounded explanation
    # --------------------------------------------------------

    explanation = (
        f"Recommendation: {occupation} is the final Top-1 career "
        f"recommendation for {candidate_id}, with an overall hybrid "
        f"recommendation score of {recommendation_percentage:.2f}%. "
        f"The recommendation combines candidate skill alignment "
        f"({skill_percentage:.2f}%), semantic similarity "
        f"({semantic_percentage:.2f}%), Canadian labour-demand evidence "
        f"({demand_percentage:.2f}%), and education alignment "
        f"({education_percentage:.2f}%). "
    )

    if missing_skill_count > 0:

        explanation += (
            f"The prescriptive skill-gap analysis identified "
            f"{missing_skill_count} missing skill(s) for this occupation. "
            f"The highest-priority development areas are: "
            f"{prioritized_skills}. "
            f"Recommended action: {development_action}. "
        )

    else:

        explanation += (
            "No priority skill gaps were identified for this occupation, "
            "so the candidate should maintain and strengthen existing "
            "occupational skills. "
        )

    explanation += (
        f"The explanation is grounded in the project's RAG knowledge base "
        f"using evidence from: {source_types}. "
        f"Supporting document IDs: {source_ids}."
    )

    return explanation


# ------------------------------------------------------------
# Regenerate explanations for all 16 candidates
# ------------------------------------------------------------

regenerated_rows = []

for row in rag_regeneration_targets.itertuples(index=False):

    regenerated_explanation = regenerate_grounded_explanation(
        row.candidate_id,
        row.top_recommended_occupation
    )

    regenerated_rows.append({
        "candidate_id": row.candidate_id,
        "recommended_occupation":
            row.top_recommended_occupation,
        "regenerated_rag_explanation":
            regenerated_explanation
    })


regenerated_rag_explanations = pd.DataFrame(
    regenerated_rows
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — REGENERATED RAG EXPLANATIONS")
print("=" * 70)

print("\nShape:")
print(regenerated_rag_explanations.shape)

print("\nUnique candidates:")
print(
    regenerated_rag_explanations[
        "candidate_id"
    ].nunique()
)

print("\nMissing regenerated explanations:")
print(
    regenerated_rag_explanations[
        "regenerated_rag_explanation"
    ].isna().sum()
)

print("\nOccupation alignment check:")

regen_validation = (
    regenerated_rag_explanations
    .merge(
        top1_candidate_summary[
            [
                "candidate_id",
                "top_recommended_occupation"
            ]
        ],
        on="candidate_id",
        how="left"
    )
)

regen_validation["occupation_match"] = (
    regen_validation["recommended_occupation"]
    ==
    regen_validation["top_recommended_occupation"]
)

print(
    regen_validation[
        "occupation_match"
    ].value_counts()
)

print("\nSample regenerated explanations:")

for _, row in regenerated_rag_explanations.head(3).iterrows():

    print("\n" + "-" * 70)
    print(row["candidate_id"])
    print(row["recommended_occupation"])
    print("-" * 70)

    print(row["regenerated_rag_explanation"])


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(regenerated_rag_explanations) == 16

assert (
    regenerated_rag_explanations[
        "candidate_id"
    ].nunique()
    == 16
)

assert (
    regenerated_rag_explanations[
        "regenerated_rag_explanation"
    ].notna()
    .all()
)

assert regen_validation[
    "occupation_match"
].all()

print(
    "\n✅ All 16 RAG explanations regenerated for the "
    "correct final-hybrid Top-1 occupations."
)

WEEK 8 — REGENERATED RAG EXPLANATIONS

Shape:
(16, 3)

Unique candidates:
16

Missing regenerated explanations:
0

Occupation alignment check:
occupation_match
True    16
Name: count, dtype: int64

Sample regenerated explanations:

----------------------------------------------------------------------
Candidate_002
Software Developer
----------------------------------------------------------------------
Recommendation: Software Developer is the final Top-1 career recommendation for Candidate_002, with an overall hybrid recommendation score of 78.25%. The recommendation combines candidate skill alignment (81.90%), semantic similarity (100.00%), Canadian labour-demand evidence (22.91%), and education alignment (100.00%). The prescriptive skill-gap analysis identified 5 missing skill(s) for this occupation. The highest-priority development areas are: Active Listening (52.72%); Speaking (49.60%); Monitoring (48.16%). Recommended action: Develop priority skills: Active Listening, Speaking, 

In [35]:
# ============================================================
# WEEK 8 — CELL 8F
# CREATE FINAL 63-CANDIDATE DEPLOYMENT RAG TABLE
# ============================================================

# ------------------------------------------------------------
# Start from safe RAG deployment table
# ------------------------------------------------------------

final_deployment_rag = deployment_rag_safe.copy()

# ------------------------------------------------------------
# Merge regenerated explanations
# ------------------------------------------------------------

final_deployment_rag = final_deployment_rag.merge(
    regenerated_rag_explanations[
        [
            "candidate_id",
            "recommended_occupation",
            "regenerated_rag_explanation"
        ]
    ],
    on="candidate_id",
    how="left"
)

# ------------------------------------------------------------
# Use original explanation when already valid,
# otherwise use regenerated explanation
# ------------------------------------------------------------

final_deployment_rag["final_rag_explanation"] = (
    final_deployment_rag[
        "deployment_rag_explanation"
    ].fillna(
        final_deployment_rag[
            "regenerated_rag_explanation"
        ]
    )
)

# ------------------------------------------------------------
# Final deployment source status
# ------------------------------------------------------------

final_deployment_rag["rag_explanation_source"] = np.where(
    final_deployment_rag[
        "rag_occupation_match"
    ],
    "Original aligned Week 7 RAG explanation",
    "Regenerated for final-hybrid Top-1 occupation"
)

final_deployment_rag["final_rag_status"] = np.where(
    final_deployment_rag[
        "final_rag_explanation"
    ].notna(),
    "Ready",
    "Missing"
)

# ------------------------------------------------------------
# Keep clean final columns
# ------------------------------------------------------------

final_deployment_rag = final_deployment_rag[
    [
        "candidate_id",
        "top_recommended_occupation",
        "final_rag_explanation",
        "rag_explanation_source",
        "final_rag_status"
    ]
].copy()

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — FINAL DEPLOYMENT RAG TABLE")
print("=" * 70)

print("\nShape:")
print(final_deployment_rag.shape)

print("\nUnique candidates:")
print(final_deployment_rag["candidate_id"].nunique())

print("\nFinal RAG status:")
display(
    final_deployment_rag[
        "final_rag_status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="candidate_count")
)

print("\nExplanation source:")
display(
    final_deployment_rag[
        "rag_explanation_source"
    ]
    .value_counts()
    .rename_axis("source")
    .reset_index(name="candidate_count")
)

print("\nMissing final explanations:")
print(
    final_deployment_rag[
        "final_rag_explanation"
    ].isna().sum()
)

print("\nSample:")
display(final_deployment_rag.head(10))

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(final_deployment_rag) == 63

assert (
    final_deployment_rag[
        "candidate_id"
    ].nunique()
    == 63
)

assert (
    final_deployment_rag[
        "final_rag_explanation"
    ].notna()
    .all()
)

assert (
    final_deployment_rag[
        "final_rag_status"
    ]
    .eq("Ready")
    .all()
)

print(
    "\n✅ All 63 candidates now have deployment-ready, "
    "Top-1-aligned RAG explanations."
)

WEEK 8 — FINAL DEPLOYMENT RAG TABLE

Shape:
(63, 5)

Unique candidates:
63

Final RAG status:


,status,candidate_count
0,Ready,63



Explanation source:


,source,candidate_count
0,Original aligned Week 7 RAG explanation,47
1,Regenerated for final-hybrid Top-1 occupation,16



Missing final explanations:
0

Sample:


,candidate_id,top_recommended_occupation,final_rag_explanation,rag_explanation_source,final_rag_status
0,Candidate_001,Software Developer,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready
1,Candidate_002,Software Developer,Recommendation: Software Developer is the fina...,Regenerated for final-hybrid Top-1 occupation,Ready
2,Candidate_003,Administrative Assistant,Recommendation: Administrative Assistant is th...,Original aligned Week 7 RAG explanation,Ready
3,Candidate_004,Software Developer,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready
4,Candidate_005,Software Developer,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready
5,Candidate_006,Software Developer,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready
6,Candidate_007,Food Service Supervisor,Recommendation: Food Service Supervisor is the...,Original aligned Week 7 RAG explanation,Ready
7,Candidate_008,Software Developer,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready
8,Candidate_009,Administrative Assistant,Recommendation: Administrative Assistant is th...,Original aligned Week 7 RAG explanation,Ready
9,Candidate_010,Food Service Supervisor,Recommendation: Food Service Supervisor is the...,Regenerated for final-hybrid Top-1 occupation,Ready



✅ All 63 candidates now have deployment-ready, Top-1-aligned RAG explanations.


In [37]:
# ============================================================
# WEEK 8 — CELL 9
# BUILD FINAL CANDIDATE DEPLOYMENT MASTER TABLE
# ============================================================

# ------------------------------------------------------------
# Start from candidate recommendation + skill-gap profile
# ------------------------------------------------------------

candidate_deployment_master = candidate_deployment_profile.copy()

# ------------------------------------------------------------
# Remove unnecessary technical index column if present
# ------------------------------------------------------------

if "index" in candidate_deployment_master.columns:
    candidate_deployment_master = (
        candidate_deployment_master
        .drop(columns=["index"])
    )

# ------------------------------------------------------------
# Merge final Top-1-aligned RAG explanations
# ------------------------------------------------------------

candidate_deployment_master = (
    candidate_deployment_master
    .merge(
        final_deployment_rag[
            [
                "candidate_id",
                "top_recommended_occupation",
                "final_rag_explanation",
                "rag_explanation_source",
                "final_rag_status"
            ]
        ],
        on=[
            "candidate_id",
            "top_recommended_occupation"
        ],
        how="left"
    )
)

# ------------------------------------------------------------
# Add overall deployment readiness
# ------------------------------------------------------------

candidate_deployment_master["deployment_status"] = np.where(
    (
        candidate_deployment_master[
            "top_recommended_occupation"
        ].notna()
    )
    &
    (
        candidate_deployment_master[
            "final_rag_explanation"
        ].notna()
    ),
    "Ready",
    "Incomplete"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — FINAL CANDIDATE DEPLOYMENT MASTER")
print("=" * 70)

print("\nShape:")
print(candidate_deployment_master.shape)

print("\nUnique candidates:")
print(
    candidate_deployment_master[
        "candidate_id"
    ].nunique()
)

print("\nDuplicate candidate IDs:")
print(
    candidate_deployment_master[
        "candidate_id"
    ].duplicated().sum()
)

print("\nDeployment status:")
display(
    candidate_deployment_master[
        "deployment_status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="candidate_count")
)

print("\nMissing Top-1 occupations:")
print(
    candidate_deployment_master[
        "top_recommended_occupation"
    ].isna().sum()
)

print("\nMissing RAG explanations:")
print(
    candidate_deployment_master[
        "final_rag_explanation"
    ].isna().sum()
)

print("\nColumns:")
print(
    candidate_deployment_master.columns.tolist()
)

print("\nSample:")
display(
    candidate_deployment_master.head(10)
)

# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert len(candidate_deployment_master) == 63

assert (
    candidate_deployment_master[
        "candidate_id"
    ].nunique()
    == 63
)

assert (
    candidate_deployment_master[
        "candidate_id"
    ].duplicated().sum()
    == 0
)

assert (
    candidate_deployment_master[
        "deployment_status"
    ].eq("Ready").all()
)

assert (
    candidate_deployment_master[
        "final_rag_explanation"
    ].notna().all()
)

print(
    "\n✅ Final 63-candidate deployment master table "
    "created successfully."
)

WEEK 8 — FINAL CANDIDATE DEPLOYMENT MASTER

Shape:
(63, 19)

Unique candidates:
63

Duplicate candidate IDs:
0

Deployment status:


,status,candidate_count
0,Ready,63



Missing Top-1 occupations:
0

Missing RAG explanations:
0

Columns:
['candidate_id', 'top_recommended_occupation', 'top_recommendation_score', 'top_recommendation_percentage', 'top_skill_percentage', 'top_semantic_percentage', 'top_demand_percentage', 'top_education_percentage', 'missing_skill_count', 'prioritized_skill_gaps', 'highest_priority_skill', 'highest_skill_priority_percentage', 'average_skill_priority_percentage', 'skill_development_action', 'skill_gap_status', 'final_rag_explanation', 'rag_explanation_source', 'final_rag_status', 'deployment_status']

Sample:


,candidate_id,top_recommended_occupation,top_recommendation_score,top_recommendation_percentage,top_skill_percentage,top_semantic_percentage,top_demand_percentage,top_education_percentage,missing_skill_count,prioritized_skill_gaps,highest_priority_skill,highest_skill_priority_percentage,average_skill_priority_percentage,skill_development_action,skill_gap_status,final_rag_explanation,rag_explanation_source,final_rag_status,deployment_status
0,Candidate_001,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,3,Active Listening (52.72%); Speaking (49.60%); ...,Active Listening,52.72,50.16,"Develop priority skills: Active Listening, Spe...",Skill development recommended,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready,Ready
1,Candidate_002,Software Developer,0.782459,78.25,81.90,100.00,22.91,100.0,5,Active Listening (52.72%); Speaking (49.60%); ...,Active Listening,52.72,45.76,"Develop priority skills: Active Listening, Spe...",Skill development recommended,Recommendation: Software Developer is the fina...,Regenerated for final-hybrid Top-1 occupation,Ready,Ready
2,Candidate_003,Administrative Assistant,0.743337,74.33,100.00,58.76,43.85,100.0,2,Mathematics (39.26%); Science (23.54%),Mathematics,39.26,31.40,"Develop priority skills: Mathematics, Science",Skill development recommended,Recommendation: Administrative Assistant is th...,Original aligned Week 7 RAG explanation,Ready,Ready
3,Candidate_004,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,2,Active Listening (52.72%); Speaking (49.60%),Active Listening,52.72,51.16,"Develop priority skills: Active Listening, Spe...",Skill development recommended,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready,Ready
4,Candidate_005,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,5,Active Listening (52.72%); Writing (50.38%); S...,Active Listening,52.72,46.80,"Develop priority skills: Active Listening, Wri...",Skill development recommended,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready,Ready
5,Candidate_006,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,1,Monitoring (48.16%),Monitoring,48.16,48.16,Develop priority skills: Monitoring,Skill development recommended,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready,Ready
6,Candidate_007,Food Service Supervisor,0.743113,74.31,100.00,44.70,68.33,100.0,0,No priority skill gaps identified,None,NaN,NaN,Maintain and strengthen existing skills for th...,No priority skill gaps identified,Recommendation: Food Service Supervisor is the...,Original aligned Week 7 RAG explanation,Ready,Ready
7,Candidate_008,Software Developer,0.845817,84.58,100.00,100.00,22.91,100.0,3,Active Listening (52.72%); Writing (50.38%); S...,Active Listening,52.72,50.90,"Develop priority skills: Active Listening, Wri...",Skill development recommended,Recommendation: Software Developer is the Top-...,Original aligned Week 7 RAG explanation,Ready,Ready
8,Candidate_009,Administrative Assistant,0.741840,74.18,100.00,58.33,43.85,100.0,8,Active Listening (64.04%); Speaking (63.26%); ...,Active Listening,64.04,49.68,"Develop priority skills: Active Listening, Spe...",Skill development recommended,Recommendation: Administrative Assistant is th...,Original aligned Week 7 RAG explanation,Ready,Ready
9,Candidate_010,Food Service Supervisor,0.721099,72.11,92.66,45.75,68.33,100.0,1,Science (33.33%),Science,33.33,33.33,Develop priority skills: Science,Skill development recommended,Recommendation: Food Service Supervisor is the...,Regenerated for final-hybrid Top-1 occupation,Ready,Ready



✅ Final 63-candidate deployment master table created successfully.


In [39]:
# ============================================================
# WEEK 8 — CELL 10
# BUILD STREAMLIT-READY TOP-5 RECOMMENDATION TABLE
# ============================================================

streamlit_top5_recommendations = top5_recommendations.copy()

# ------------------------------------------------------------
# Create user-friendly display labels
# ------------------------------------------------------------

streamlit_top5_recommendations["recommendation_label"] = (
    "Rank "
    + streamlit_top5_recommendations[
        "recommendation_rank"
    ].astype(str)
    + ": "
    + streamlit_top5_recommendations[
        "recommended_occupation"
    ]
)

# ------------------------------------------------------------
# Create concise recommendation summary
# ------------------------------------------------------------

streamlit_top5_recommendations["recommendation_summary"] = (
    streamlit_top5_recommendations[
        "recommended_occupation"
    ]
    + " — Overall Match: "
    + streamlit_top5_recommendations[
        "recommendation_percentage"
    ].round(2).astype(str)
    + "% | Skill: "
    + streamlit_top5_recommendations[
        "skill_percentage"
    ].round(2).astype(str)
    + "% | Semantic: "
    + streamlit_top5_recommendations[
        "semantic_percentage"
    ].round(2).astype(str)
    + "% | Demand: "
    + streamlit_top5_recommendations[
        "demand_percentage"
    ].round(2).astype(str)
    + "% | Education: "
    + streamlit_top5_recommendations[
        "education_percentage"
    ].round(2).astype(str)
    + "%"
)

# ------------------------------------------------------------
# Sort for deployment
# ------------------------------------------------------------

streamlit_top5_recommendations = (
    streamlit_top5_recommendations
    .sort_values(
        [
            "candidate_id",
            "recommendation_rank"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — STREAMLIT TOP-5 RECOMMENDATION TABLE")
print("=" * 70)

print("\nShape:")
print(streamlit_top5_recommendations.shape)

print("\nUnique candidates:")
print(
    streamlit_top5_recommendations[
        "candidate_id"
    ].nunique()
)

print("\nRows per candidate:")
display(
    streamlit_top5_recommendations
    .groupby("candidate_id")
    .size()
    .value_counts()
    .rename_axis("recommendation_count")
    .reset_index(name="candidate_count")
)

print("\nRank distribution:")
display(
    streamlit_top5_recommendations[
        "recommendation_rank"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("rank")
    .reset_index(name="row_count")
)

print("\nSample:")
display(
    streamlit_top5_recommendations.head(10)
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(streamlit_top5_recommendations) == 315

assert (
    streamlit_top5_recommendations[
        "candidate_id"
    ].nunique()
    == 63
)

assert (
    streamlit_top5_recommendations
    .groupby("candidate_id")
    .size()
    .eq(5)
    .all()
)

assert set(
    streamlit_top5_recommendations[
        "recommendation_rank"
    ].unique()
) == {1, 2, 3, 4, 5}

print(
    "\n✅ Streamlit-ready Top-5 recommendation table "
    "created successfully."
)

WEEK 8 — STREAMLIT TOP-5 RECOMMENDATION TABLE

Shape:
(315, 11)

Unique candidates:
63

Rows per candidate:


,recommendation_count,candidate_count
0,5,63



Rank distribution:


,rank,row_count
0,1,63
1,2,63
2,3,63
3,4,63
4,5,63



Sample:


,candidate_id,recommended_occupation,recommendation_rank,recommendation_score,recommendation_percentage,skill_percentage,semantic_percentage,demand_percentage,education_percentage,recommendation_label,recommendation_summary
0,Candidate_001,Software Developer,1,0.845817,84.58,100.00,100.00,22.91,100.0,Rank 1: Software Developer,Software Developer — Overall Match: 84.58% | S...
1,Candidate_001,Information Technology (IT) Analyst,2,0.652990,65.30,69.68,73.10,26.64,100.0,Rank 2: Information Technology (IT) Analyst,Information Technology (IT) Analyst — Overall ...
2,Candidate_001,Bookkeeper,3,0.603429,60.34,61.28,65.46,29.92,100.0,Rank 3: Bookkeeper,Bookkeeper — Overall Match: 60.34% | Skill: 61...
3,Candidate_001,Secondary School Teacher,4,0.591983,59.20,62.29,57.50,36.36,100.0,Rank 4: Secondary School Teacher,Secondary School Teacher — Overall Match: 59.2...
4,Candidate_001,Office Administrator,5,0.465784,46.58,43.62,59.26,52.86,0.0,Rank 5: Office Administrator,Office Administrator — Overall Match: 46.58% |...
5,Candidate_002,Software Developer,1,0.782459,78.25,81.90,100.00,22.91,100.0,Rank 1: Software Developer,Software Developer — Overall Match: 78.25% | S...
6,Candidate_002,Office Manager,2,0.671262,67.13,100.00,33.01,52.86,100.0,Rank 2: Office Manager,Office Manager — Overall Match: 67.13% | Skill...
7,Candidate_002,Information Technology (IT) Analyst,3,0.634155,63.42,61.55,75.84,26.64,100.0,Rank 3: Information Technology (IT) Analyst,Information Technology (IT) Analyst — Overall ...
8,Candidate_002,Office Administrator,4,0.622124,62.21,81.98,36.99,52.86,100.0,Rank 4: Office Administrator,Office Administrator — Overall Match: 62.21% |...
9,Candidate_002,Secondary School Teacher,5,0.610059,61.01,78.23,46.73,36.36,100.0,Rank 5: Secondary School Teacher,Secondary School Teacher — Overall Match: 61.0...



✅ Streamlit-ready Top-5 recommendation table created successfully.


In [41]:
# ============================================================
# WEEK 8 — CELL 11
# BUILD OCCUPATION-LEVEL DEPLOYMENT SUMMARY
# ============================================================

occupation_deployment_summary = (
    candidate_deployment_master
    .groupby(
        "top_recommended_occupation",
        as_index=False
    )
    .agg(
        top1_candidate_count=(
            "candidate_id",
            "nunique"
        ),
        average_recommendation_percentage=(
            "top_recommendation_percentage",
            "mean"
        ),
        average_skill_percentage=(
            "top_skill_percentage",
            "mean"
        ),
        average_semantic_percentage=(
            "top_semantic_percentage",
            "mean"
        ),
        average_demand_percentage=(
            "top_demand_percentage",
            "mean"
        ),
        average_education_percentage=(
            "top_education_percentage",
            "mean"
        ),
        average_missing_skill_count=(
            "missing_skill_count",
            "mean"
        ),
        candidates_with_skill_gaps=(
            "skill_gap_status",
            lambda x: (
                x == "Skill development recommended"
            ).sum()
        )
    )
)

# ------------------------------------------------------------
# Add Top-1 recommendation share
# ------------------------------------------------------------

occupation_deployment_summary[
    "top1_candidate_share_percentage"
] = (
    occupation_deployment_summary[
        "top1_candidate_count"
    ]
    / candidate_deployment_master[
        "candidate_id"
    ].nunique()
    * 100
)

# ------------------------------------------------------------
# Round dashboard metrics
# ------------------------------------------------------------

percentage_columns = [
    "average_recommendation_percentage",
    "average_skill_percentage",
    "average_semantic_percentage",
    "average_demand_percentage",
    "average_education_percentage",
    "top1_candidate_share_percentage"
]

occupation_deployment_summary[
    percentage_columns
] = occupation_deployment_summary[
    percentage_columns
].round(2)

occupation_deployment_summary[
    "average_missing_skill_count"
] = occupation_deployment_summary[
    "average_missing_skill_count"
].round(2)

# ------------------------------------------------------------
# Sort by Top-1 recommendation frequency
# ------------------------------------------------------------

occupation_deployment_summary = (
    occupation_deployment_summary
    .sort_values(
        [
            "top1_candidate_count",
            "average_recommendation_percentage"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — OCCUPATION-LEVEL DEPLOYMENT SUMMARY")
print("=" * 70)

print("\nShape:")
print(occupation_deployment_summary.shape)

print("\nTotal Top-1 candidates represented:")
print(
    occupation_deployment_summary[
        "top1_candidate_count"
    ].sum()
)

print("\nTop-1 recommendation share total:")
print(
    round(
        occupation_deployment_summary[
            "top1_candidate_share_percentage"
        ].sum(),
        2
    ),
    "%"
)

print("\nOccupation summary:")
display(occupation_deployment_summary)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert (
    occupation_deployment_summary[
        "top1_candidate_count"
    ].sum()
    == 63
)

assert (
    occupation_deployment_summary[
        "top1_candidate_count"
    ] > 0
).all()

assert (
    occupation_deployment_summary[
        "average_recommendation_percentage"
    ].between(0, 100).all()
)

print(
    "\n✅ Occupation-level deployment summary "
    "created successfully."
)

WEEK 8 — OCCUPATION-LEVEL DEPLOYMENT SUMMARY

Shape:
(4, 10)

Total Top-1 candidates represented:
63

Top-1 recommendation share total:
100.0 %

Occupation summary:


,top_recommended_occupation,top1_candidate_count,average_recommendation_percentage,average_skill_percentage,average_semantic_percentage,average_demand_percentage,average_education_percentage,average_missing_skill_count,candidates_with_skill_gaps,top1_candidate_share_percentage
0,Software Developer,35,82.47,93.96,100.00,22.91,100.0,3.49,35,55.56
1,Administrative Assistant,13,71.97,100.00,52.00,43.85,100.0,2.54,12,20.63
2,Food Service Supervisor,11,74.36,96.66,48.18,68.33,100.0,0.45,5,17.46
3,Office Manager,4,71.40,100.00,45.21,52.86,100.0,3.50,4,6.35



✅ Occupation-level deployment summary created successfully.


In [43]:
# ============================================================
# WEEK 8 — CELL 12
# BUILD FINAL PROJECT KPI SUMMARY
# ============================================================

total_candidates = (
    candidate_deployment_master[
        "candidate_id"
    ].nunique()
)

total_occupations = (
    final_hybrid[
        "selected_occupation"
    ].nunique()
)

total_candidate_occupation_pairs = len(final_hybrid)

deployment_ready_candidates = (
    candidate_deployment_master[
        "deployment_status"
    ]
    .eq("Ready")
    .sum()
)

candidates_with_skill_gaps = (
    candidate_deployment_master[
        "skill_gap_status"
    ]
    .eq("Skill development recommended")
    .sum()
)

candidates_without_priority_gaps = (
    candidate_deployment_master[
        "skill_gap_status"
    ]
    .eq("No priority skill gaps identified")
    .sum()
)

average_top1_recommendation = (
    candidate_deployment_master[
        "top_recommendation_percentage"
    ].mean()
)

average_top1_skill = (
    candidate_deployment_master[
        "top_skill_percentage"
    ].mean()
)

average_top1_semantic = (
    candidate_deployment_master[
        "top_semantic_percentage"
    ].mean()
)

average_top1_demand = (
    candidate_deployment_master[
        "top_demand_percentage"
    ].mean()
)

average_missing_skills = (
    candidate_deployment_master[
        "missing_skill_count"
    ].mean()
)

rag_ready_count = (
    candidate_deployment_master[
        "final_rag_status"
    ]
    .eq("Ready")
    .sum()
)

rag_original_count = (
    candidate_deployment_master[
        "rag_explanation_source"
    ]
    .eq(
        "Original aligned Week 7 RAG explanation"
    )
    .sum()
)

rag_regenerated_count = (
    candidate_deployment_master[
        "rag_explanation_source"
    ]
    .eq(
        "Regenerated for final-hybrid Top-1 occupation"
    )
    .sum()
)

most_common_top1 = (
    candidate_deployment_master[
        "top_recommended_occupation"
    ]
    .value_counts()
    .idxmax()
)

most_common_top1_count = (
    candidate_deployment_master[
        "top_recommended_occupation"
    ]
    .value_counts()
    .max()
)

# ------------------------------------------------------------
# Build KPI table
# ------------------------------------------------------------

project_kpi_summary = pd.DataFrame(
    [
        {
            "kpi": "Total Candidates",
            "value": total_candidates
        },
        {
            "kpi": "Occupations Evaluated",
            "value": total_occupations
        },
        {
            "kpi": "Candidate-Occupation Pairs",
            "value": total_candidate_occupation_pairs
        },
        {
            "kpi": "Deployment-Ready Candidates",
            "value": deployment_ready_candidates
        },
        {
            "kpi": "Candidates With Priority Skill Gaps",
            "value": candidates_with_skill_gaps
        },
        {
            "kpi": "Candidates Without Priority Skill Gaps",
            "value": candidates_without_priority_gaps
        },
        {
            "kpi": "Average Top-1 Recommendation %",
            "value": round(
                average_top1_recommendation, 2
            )
        },
        {
            "kpi": "Average Top-1 Skill %",
            "value": round(
                average_top1_skill, 2
            )
        },
        {
            "kpi": "Average Top-1 Semantic %",
            "value": round(
                average_top1_semantic, 2
            )
        },
        {
            "kpi": "Average Top-1 Demand %",
            "value": round(
                average_top1_demand, 2
            )
        },
        {
            "kpi": "Average Missing Skills per Candidate",
            "value": round(
                average_missing_skills, 2
            )
        },
        {
            "kpi": "RAG Explanations Ready",
            "value": rag_ready_count
        },
        {
            "kpi": "Original Aligned RAG Explanations",
            "value": rag_original_count
        },
        {
            "kpi": "Regenerated RAG Explanations",
            "value": rag_regenerated_count
        },
        {
            "kpi": "Most Common Top-1 Occupation",
            "value": most_common_top1
        },
        {
            "kpi": "Candidates Receiving Most Common Top-1",
            "value": most_common_top1_count
        }
    ]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — FINAL PROJECT KPI SUMMARY")
print("=" * 70)

display(project_kpi_summary)

print("\nCore validation:")

print(
    "Deployment ready:",
    deployment_ready_candidates,
    "/",
    total_candidates
)

print(
    "RAG ready:",
    rag_ready_count,
    "/",
    total_candidates
)

print(
    "Skill-gap coverage:",
    candidates_with_skill_gaps
    + candidates_without_priority_gaps,
    "/",
    total_candidates
)

print(
    "RAG source coverage:",
    rag_original_count
    + rag_regenerated_count,
    "/",
    total_candidates
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert total_candidates == 63
assert total_occupations == 15
assert total_candidate_occupation_pairs == 945

assert deployment_ready_candidates == 63
assert rag_ready_count == 63

assert (
    candidates_with_skill_gaps
    + candidates_without_priority_gaps
    == 63
)

assert (
    rag_original_count
    + rag_regenerated_count
    == 63
)

print(
    "\n✅ Final project KPI summary created successfully."
)

WEEK 8 — FINAL PROJECT KPI SUMMARY


,kpi,value
0,Total Candidates,63
1,Occupations Evaluated,15
2,Candidate-Occupation Pairs,945
3,Deployment-Ready Candidates,63
4,Candidates With Priority Skill Gaps,56
5,Candidates Without Priority Skill Gaps,7
6,Average Top-1 Recommendation %,78.18
7,Average Top-1 Skill %,96.06
8,Average Top-1 Semantic %,77.57
9,Average Top-1 Demand %,37.06



Core validation:
Deployment ready: 63 / 63
RAG ready: 63 / 63
Skill-gap coverage: 63 / 63
RAG source coverage: 63 / 63

✅ Final project KPI summary created successfully.


In [45]:
# ============================================================
# WEEK 8 — CELL 13
# BUILD FINAL BUSINESS / RESEARCH INSIGHTS TABLE
# ============================================================

# ------------------------------------------------------------
# Identify key occupation-level findings
# ------------------------------------------------------------

most_recommended_row = (
    occupation_deployment_summary
    .sort_values(
        "top1_candidate_count",
        ascending=False
    )
    .iloc[0]
)

highest_demand_row = (
    occupation_deployment_summary
    .sort_values(
        "average_demand_percentage",
        ascending=False
    )
    .iloc[0]
)

highest_match_row = (
    occupation_deployment_summary
    .sort_values(
        "average_recommendation_percentage",
        ascending=False
    )
    .iloc[0]
)

lowest_gap_row = (
    occupation_deployment_summary
    .sort_values(
        "average_missing_skill_count",
        ascending=True
    )
    .iloc[0]
)

highest_gap_row = (
    occupation_deployment_summary
    .sort_values(
        "average_missing_skill_count",
        ascending=False
    )
    .iloc[0]
)

# ------------------------------------------------------------
# Create final insights
# ------------------------------------------------------------

final_project_insights = pd.DataFrame(
    [
        {
            "insight_id": "INSIGHT_01",
            "category": "Recommendation Concentration",
            "finding": (
                f"{most_recommended_row['top_recommended_occupation']} "
                f"is the most frequent Top-1 recommendation, selected for "
                f"{int(most_recommended_row['top1_candidate_count'])} "
                f"of 63 candidates "
                f"({most_recommended_row['top1_candidate_share_percentage']:.2f}%)."
            ),
            "implication": (
                "The candidate dataset shows a strong concentration toward "
                "this occupation under the current hybrid recommendation model."
            )
        },

        {
            "insight_id": "INSIGHT_02",
            "category": "Recommendation Strength",
            "finding": (
                f"The average Top-1 recommendation score across all candidates "
                f"is {average_top1_recommendation:.2f}%."
            ),
            "implication": (
                "The final hybrid model generally produces relatively strong "
                "Top-1 matches across the evaluated candidate profiles."
            )
        },

        {
            "insight_id": "INSIGHT_03",
            "category": "Skill Alignment",
            "finding": (
                f"Average Top-1 skill alignment is "
                f"{average_top1_skill:.2f}%."
            ),
            "implication": (
                "Candidate skill evidence contributes strongly to the final "
                "career recommendations."
            )
        },

        {
            "insight_id": "INSIGHT_04",
            "category": "Skill Development",
            "finding": (
                f"{candidates_with_skill_gaps} of {total_candidates} "
                f"candidates have at least one priority skill gap, with an "
                f"average of {average_missing_skills:.2f} missing skills "
                f"per candidate."
            ),
            "implication": (
                "Prescriptive recommendations remain important even when "
                "overall career-match scores are high."
            )
        },

        {
            "insight_id": "INSIGHT_05",
            "category": "Labour Demand",
            "finding": (
                f"{highest_demand_row['top_recommended_occupation']} has the "
                f"highest average labour-demand score among Top-1 occupations "
                f"at {highest_demand_row['average_demand_percentage']:.2f}%."
            ),
            "implication": (
                "Labour-market attractiveness differs across recommended "
                "careers and should be considered alongside candidate fit."
            )
        },

        {
            "insight_id": "INSIGHT_06",
            "category": "Highest Average Match",
            "finding": (
                f"{highest_match_row['top_recommended_occupation']} has the "
                f"highest average recommendation score among Top-1 occupations "
                f"at {highest_match_row['average_recommendation_percentage']:.2f}%."
            ),
            "implication": (
                "This occupation shows the strongest average combined fit "
                "within the current candidate cohort."
            )
        },

        {
            "insight_id": "INSIGHT_07",
            "category": "Skill-Gap Burden",
            "finding": (
                f"{lowest_gap_row['top_recommended_occupation']} has the "
                f"lowest average missing-skill count "
                f"({lowest_gap_row['average_missing_skill_count']:.2f}), "
                f"while {highest_gap_row['top_recommended_occupation']} has "
                f"the highest "
                f"({highest_gap_row['average_missing_skill_count']:.2f})."
            ),
            "implication": (
                "Different recommended occupations require substantially "
                "different levels of candidate skill development."
            )
        },

        {
            "insight_id": "INSIGHT_08",
            "category": "RAG Explainability",
            "finding": (
                f"All {rag_ready_count} candidates have deployment-ready "
                f"RAG explanations aligned to their final Top-1 occupation."
            ),
            "implication": (
                "The deployment layer provides interpretable recommendation "
                "evidence rather than presenting ranking scores alone."
            )
        },

        {
            "insight_id": "INSIGHT_09",
            "category": "RAG Quality Control",
            "finding": (
                f"{rag_original_count} Week 7 explanations were already "
                f"aligned with the final Top-1 recommendation, while "
                f"{rag_regenerated_count} were regenerated after occupation "
                f"alignment validation."
            ),
            "implication": (
                "Explicit occupation-level validation prevents explanations "
                "from being attached to the wrong final recommendation."
            )
        },

        {
            "insight_id": "INSIGHT_10",
            "category": "Deployment Coverage",
            "finding": (
                f"All {deployment_ready_candidates} of {total_candidates} "
                f"candidate profiles contain a Top-1 recommendation, "
                f"prescriptive skill-gap result, and aligned RAG explanation."
            ),
            "implication": (
                "The analytical pipeline has complete candidate-level "
                "coverage for the planned deployment interface."
            )
        }
    ]
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — FINAL BUSINESS / RESEARCH INSIGHTS")
print("=" * 70)

print("\nShape:")
print(final_project_insights.shape)

display(final_project_insights)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(final_project_insights) == 10

assert (
    final_project_insights[
        "finding"
    ].notna().all()
)

assert (
    final_project_insights[
        "implication"
    ].notna().all()
)

print(
    "\n✅ Final business/research insights table "
    "created successfully."
)

WEEK 8 — FINAL BUSINESS / RESEARCH INSIGHTS

Shape:
(10, 4)


,insight_id,category,finding,implication
0,INSIGHT_01,Recommendation Concentration,Software Developer is the most frequent Top-1 ...,The candidate dataset shows a strong concentra...
1,INSIGHT_02,Recommendation Strength,The average Top-1 recommendation score across ...,The final hybrid model generally produces rela...
2,INSIGHT_03,Skill Alignment,Average Top-1 skill alignment is 96.06%.,Candidate skill evidence contributes strongly ...
3,INSIGHT_04,Skill Development,56 of 63 candidates have at least one priority...,Prescriptive recommendations remain important ...
4,INSIGHT_05,Labour Demand,Food Service Supervisor has the highest averag...,Labour-market attractiveness differs across re...
5,INSIGHT_06,Highest Average Match,Software Developer has the highest average rec...,This occupation shows the strongest average co...
6,INSIGHT_07,Skill-Gap Burden,Food Service Supervisor has the lowest average...,Different recommended occupations require subs...
7,INSIGHT_08,RAG Explainability,All 63 candidates have deployment-ready RAG ex...,The deployment layer provides interpretable re...
8,INSIGHT_09,RAG Quality Control,47 Week 7 explanations were already aligned wi...,Explicit occupation-level validation prevents ...
9,INSIGHT_10,Deployment Coverage,All 63 of 63 candidate profiles contain a Top-...,The analytical pipeline has complete candidate...



✅ Final business/research insights table created successfully.


In [47]:
# ============================================================
# WEEK 8 — CELL 14
# CREATE DEPLOYMENT CONFIGURATION / ARTIFACT REGISTER
# ============================================================

deployment_artifact_register = pd.DataFrame(
    [
        {
            "artifact_id": "DEPLOY_01",
            "artifact_name": "Candidate Deployment Master",
            "python_object": "candidate_deployment_master",
            "planned_file": "week8_candidate_deployment_master.csv",
            "purpose": (
                "Primary candidate-level deployment table containing "
                "Top-1 recommendation, skill-gap guidance, and aligned "
                "RAG explanation."
            ),
            "expected_rows": 63
        },
        {
            "artifact_id": "DEPLOY_02",
            "artifact_name": "Streamlit Top-5 Recommendations",
            "python_object": "streamlit_top5_recommendations",
            "planned_file": "week8_streamlit_top5_recommendations.csv",
            "purpose": (
                "Provides five ranked career recommendations for each "
                "candidate for the Streamlit recommendation interface."
            ),
            "expected_rows": 315
        },
        {
            "artifact_id": "DEPLOY_03",
            "artifact_name": "Final Deployment RAG",
            "python_object": "final_deployment_rag",
            "planned_file": "week8_final_deployment_rag.csv",
            "purpose": (
                "Stores final Top-1-aligned grounded explanations for "
                "all candidates."
            ),
            "expected_rows": 63
        },
        {
            "artifact_id": "DEPLOY_04",
            "artifact_name": "Occupation Deployment Summary",
            "python_object": "occupation_deployment_summary",
            "planned_file": "week8_occupation_deployment_summary.csv",
            "purpose": (
                "Occupation-level summary for dashboard visualization "
                "and final analytical reporting."
            ),
            "expected_rows": len(
                occupation_deployment_summary
            )
        },
        {
            "artifact_id": "DEPLOY_05",
            "artifact_name": "Project KPI Summary",
            "python_object": "project_kpi_summary",
            "planned_file": "week8_project_kpi_summary.csv",
            "purpose": (
                "Stores final project and deployment KPIs."
            ),
            "expected_rows": len(
                project_kpi_summary
            )
        },
        {
            "artifact_id": "DEPLOY_06",
            "artifact_name": "Final Project Insights",
            "python_object": "final_project_insights",
            "planned_file": "week8_final_project_insights.csv",
            "purpose": (
                "Stores final evidence-based business and research "
                "findings with their implications."
            ),
            "expected_rows": len(
                final_project_insights
            )
        }
    ]
)

# ------------------------------------------------------------
# Deployment configuration
# ------------------------------------------------------------

deployment_configuration = pd.DataFrame(
    [
        {
            "configuration": "Application",
            "value": "Streamlit"
        },
        {
            "configuration": "Recommendation Type",
            "value": "Content-based hybrid ranking"
        },
        {
            "configuration": "Candidates",
            "value": 63
        },
        {
            "configuration": "Occupations Evaluated",
            "value": 15
        },
        {
            "configuration": "Recommendations Displayed",
            "value": "Top 5 per candidate"
        },
        {
            "configuration": "Primary Recommendation",
            "value": "Final hybrid Top-1 occupation"
        },
        {
            "configuration": "Prescriptive Layer",
            "value": "Top-1 occupation skill-gap analysis"
        },
        {
            "configuration": "Explainability Layer",
            "value": "Top-1-aligned grounded RAG explanation"
        },
        {
            "configuration": "RAG Knowledge Base Documents",
            "value": len(rag_knowledge_base)
        },
        {
            "configuration": "Deployment Ready Candidates",
            "value": 63
        },
        {
            "configuration": "Human Evaluation Status",
            "value": "Pending genuine human review"
        }
    ]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — DEPLOYMENT CONFIGURATION")
print("=" * 70)

print("\nDeployment artifact register:")
display(deployment_artifact_register)

print("\nDeployment configuration:")
display(deployment_configuration)

print("\nNumber of planned deployment artifacts:")
print(len(deployment_artifact_register))

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(deployment_artifact_register) == 6

assert (
    deployment_artifact_register[
        "artifact_id"
    ].nunique()
    == 6
)

assert (
    deployment_artifact_register[
        "planned_file"
    ].nunique()
    == 6
)

assert (
    deployment_configuration.loc[
        deployment_configuration[
            "configuration"
        ] == "Human Evaluation Status",
        "value"
    ].iloc[0]
    == "Pending genuine human review"
)

print(
    "\n✅ Deployment configuration and artifact register "
    "created successfully."
)

WEEK 8 — DEPLOYMENT CONFIGURATION

Deployment artifact register:


,artifact_id,artifact_name,python_object,planned_file,purpose,expected_rows
0,DEPLOY_01,Candidate Deployment Master,candidate_deployment_master,week8_candidate_deployment_master.csv,Primary candidate-level deployment table conta...,63
1,DEPLOY_02,Streamlit Top-5 Recommendations,streamlit_top5_recommendations,week8_streamlit_top5_recommendations.csv,Provides five ranked career recommendations fo...,315
2,DEPLOY_03,Final Deployment RAG,final_deployment_rag,week8_final_deployment_rag.csv,Stores final Top-1-aligned grounded explanatio...,63
3,DEPLOY_04,Occupation Deployment Summary,occupation_deployment_summary,week8_occupation_deployment_summary.csv,Occupation-level summary for dashboard visuali...,4
4,DEPLOY_05,Project KPI Summary,project_kpi_summary,week8_project_kpi_summary.csv,Stores final project and deployment KPIs.,16
5,DEPLOY_06,Final Project Insights,final_project_insights,week8_final_project_insights.csv,Stores final evidence-based business and resea...,10



Deployment configuration:


,configuration,value
0,Application,Streamlit
1,Recommendation Type,Content-based hybrid ranking
2,Candidates,63
3,Occupations Evaluated,15
4,Recommendations Displayed,Top 5 per candidate
5,Primary Recommendation,Final hybrid Top-1 occupation
6,Prescriptive Layer,Top-1 occupation skill-gap analysis
7,Explainability Layer,Top-1-aligned grounded RAG explanation
8,RAG Knowledge Base Documents,707
9,Deployment Ready Candidates,63



Number of planned deployment artifacts:
6

✅ Deployment configuration and artifact register created successfully.


In [49]:
# ============================================================
# WEEK 8 — CELL 15
# FINAL REPRODUCIBILITY AND INTEGRITY AUDIT
# ============================================================

validation_rows = []

# ------------------------------------------------------------
# 1. Candidate coverage
# ------------------------------------------------------------

validation_rows.append({
    "validation_check": "Deployment master has 63 candidates",
    "expected": 63,
    "actual": candidate_deployment_master[
        "candidate_id"
    ].nunique(),
    "status": (
        "PASS"
        if candidate_deployment_master[
            "candidate_id"
        ].nunique() == 63
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 2. Candidate-occupation pair coverage
# ------------------------------------------------------------

validation_rows.append({
    "validation_check": "Final hybrid has 945 candidate-occupation pairs",
    "expected": 945,
    "actual": len(final_hybrid),
    "status": (
        "PASS"
        if len(final_hybrid) == 945
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 3. Occupation coverage
# ------------------------------------------------------------

validation_rows.append({
    "validation_check": "Final hybrid evaluates 15 occupations",
    "expected": 15,
    "actual": final_hybrid[
        "selected_occupation"
    ].nunique(),
    "status": (
        "PASS"
        if final_hybrid[
            "selected_occupation"
        ].nunique() == 15
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 4. Top-5 coverage
# ------------------------------------------------------------

validation_rows.append({
    "validation_check": "Top-5 table has 315 rows",
    "expected": 315,
    "actual": len(
        streamlit_top5_recommendations
    ),
    "status": (
        "PASS"
        if len(
            streamlit_top5_recommendations
        ) == 315
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 5. Exactly five recommendations per candidate
# ------------------------------------------------------------

five_per_candidate = (
    streamlit_top5_recommendations
    .groupby("candidate_id")
    .size()
    .eq(5)
    .all()
)

validation_rows.append({
    "validation_check": "Every candidate has exactly 5 recommendations",
    "expected": True,
    "actual": five_per_candidate,
    "status": (
        "PASS"
        if five_per_candidate
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 6. Top-1 occupation alignment
# ------------------------------------------------------------

top1_alignment = (
    candidate_deployment_master[
        [
            "candidate_id",
            "top_recommended_occupation"
        ]
    ]
    .merge(
        streamlit_top5_recommendations[
            streamlit_top5_recommendations[
                "recommendation_rank"
            ] == 1
        ][
            [
                "candidate_id",
                "recommended_occupation"
            ]
        ],
        on="candidate_id",
        how="left"
    )
)

top1_alignment_pass = (
    top1_alignment[
        "top_recommended_occupation"
    ]
    ==
    top1_alignment[
        "recommended_occupation"
    ]
).all()

validation_rows.append({
    "validation_check": (
        "Deployment Top-1 matches Streamlit rank 1"
    ),
    "expected": True,
    "actual": top1_alignment_pass,
    "status": (
        "PASS"
        if top1_alignment_pass
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 7. RAG coverage
# ------------------------------------------------------------

rag_complete = (
    candidate_deployment_master[
        "final_rag_explanation"
    ].notna().all()
)

validation_rows.append({
    "validation_check": "All candidates have RAG explanations",
    "expected": True,
    "actual": rag_complete,
    "status": (
        "PASS"
        if rag_complete
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 8. Deployment readiness
# ------------------------------------------------------------

deployment_complete = (
    candidate_deployment_master[
        "deployment_status"
    ].eq("Ready").all()
)

validation_rows.append({
    "validation_check": "All candidates are deployment ready",
    "expected": True,
    "actual": deployment_complete,
    "status": (
        "PASS"
        if deployment_complete
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 9. Skill-gap coverage
# ------------------------------------------------------------

skill_gap_complete = (
    candidate_deployment_master[
        "skill_gap_status"
    ].notna().all()
)

validation_rows.append({
    "validation_check": (
        "All candidates have skill-gap status"
    ),
    "expected": True,
    "actual": skill_gap_complete,
    "status": (
        "PASS"
        if skill_gap_complete
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 10. RAG knowledge-base integrity
# ------------------------------------------------------------

validation_rows.append({
    "validation_check": "RAG knowledge base has 707 documents",
    "expected": 707,
    "actual": len(rag_knowledge_base),
    "status": (
        "PASS"
        if len(rag_knowledge_base) == 707
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 11. RAG explanation source reconciliation
# ------------------------------------------------------------

rag_source_total = (
    candidate_deployment_master[
        "rag_explanation_source"
    ].notna().sum()
)

validation_rows.append({
    "validation_check": (
        "RAG explanation sources reconcile to 63 candidates"
    ),
    "expected": 63,
    "actual": rag_source_total,
    "status": (
        "PASS"
        if rag_source_total == 63
        else "FAIL"
    )
})

# ------------------------------------------------------------
# 12. Human evaluation remains pending
# ------------------------------------------------------------

human_evaluation_status = (
    deployment_configuration.loc[
        deployment_configuration[
            "configuration"
        ] == "Human Evaluation Status",
        "value"
    ].iloc[0]
)

validation_rows.append({
    "validation_check": (
        "Human evaluation status is correctly documented"
    ),
    "expected": "Pending genuine human review",
    "actual": human_evaluation_status,
    "status": (
        "PASS"
        if human_evaluation_status
        == "Pending genuine human review"
        else "FAIL"
    )
})

# ------------------------------------------------------------
# Create final validation table
# ------------------------------------------------------------

week8_integrity_validation = pd.DataFrame(
    validation_rows
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — FINAL REPRODUCIBILITY & INTEGRITY AUDIT")
print("=" * 70)

display(week8_integrity_validation)

print("\nValidation status counts:")

display(
    week8_integrity_validation[
        "status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="check_count")
)

failed_checks = week8_integrity_validation[
    week8_integrity_validation[
        "status"
    ] != "PASS"
]

print("\nFailed checks:")
print(len(failed_checks))

if len(failed_checks) > 0:
    display(failed_checks)

# ------------------------------------------------------------
# Final integrity assertion
# ------------------------------------------------------------

assert (
    week8_integrity_validation[
        "status"
    ].eq("PASS").all()
)

print(
    "\n✅ Week 8 reproducibility and integrity audit passed."
)

WEEK 8 — FINAL REPRODUCIBILITY & INTEGRITY AUDIT


,validation_check,expected,actual,status
0,Deployment master has 63 candidates,63,63,PASS
1,Final hybrid has 945 candidate-occupation pairs,945,945,PASS
2,Final hybrid evaluates 15 occupations,15,15,PASS
3,Top-5 table has 315 rows,315,315,PASS
4,Every candidate has exactly 5 recommendations,True,True,PASS
5,Deployment Top-1 matches Streamlit rank 1,True,True,PASS
6,All candidates have RAG explanations,True,True,PASS
7,All candidates are deployment ready,True,True,PASS
8,All candidates have skill-gap status,True,True,PASS
9,RAG knowledge base has 707 documents,707,707,PASS



Validation status counts:


,status,check_count
0,PASS,12



Failed checks:
0

✅ Week 8 reproducibility and integrity audit passed.


In [51]:
# ============================================================
# WEEK 8 — CELL 16
# EXPORT FINAL WEEK 8 DELIVERABLES
# ============================================================

# ------------------------------------------------------------
# Define export paths
# ------------------------------------------------------------

week8_export_files = {
    "candidate_deployment_master":
        WEEK8_OUTPUT_DIR / "week8_candidate_deployment_master.csv",

    "streamlit_top5_recommendations":
        WEEK8_OUTPUT_DIR / "week8_streamlit_top5_recommendations.csv",

    "final_deployment_rag":
        WEEK8_OUTPUT_DIR / "week8_final_deployment_rag.csv",

    "occupation_deployment_summary":
        WEEK8_OUTPUT_DIR / "week8_occupation_deployment_summary.csv",

    "project_kpi_summary":
        WEEK8_OUTPUT_DIR / "week8_project_kpi_summary.csv",

    "final_project_insights":
        WEEK8_OUTPUT_DIR / "week8_final_project_insights.csv",

    "deployment_artifact_register":
        WEEK8_OUTPUT_DIR / "week8_deployment_artifact_register.csv",

    "deployment_configuration":
        WEEK8_OUTPUT_DIR / "week8_deployment_configuration.csv",

    "integrity_validation":
        WEEK8_OUTPUT_DIR / "week8_integrity_validation.csv"
}

# ------------------------------------------------------------
# Export CSV files
# ------------------------------------------------------------

candidate_deployment_master.to_csv(
    week8_export_files["candidate_deployment_master"],
    index=False
)

streamlit_top5_recommendations.to_csv(
    week8_export_files["streamlit_top5_recommendations"],
    index=False
)

final_deployment_rag.to_csv(
    week8_export_files["final_deployment_rag"],
    index=False
)

occupation_deployment_summary.to_csv(
    week8_export_files["occupation_deployment_summary"],
    index=False
)

project_kpi_summary.to_csv(
    week8_export_files["project_kpi_summary"],
    index=False
)

final_project_insights.to_csv(
    week8_export_files["final_project_insights"],
    index=False
)

deployment_artifact_register.to_csv(
    week8_export_files["deployment_artifact_register"],
    index=False
)

deployment_configuration.to_csv(
    week8_export_files["deployment_configuration"],
    index=False
)

week8_integrity_validation.to_csv(
    week8_export_files["integrity_validation"],
    index=False
)

# ------------------------------------------------------------
# Build export validation table
# ------------------------------------------------------------

export_validation_rows = []

for artifact_name, file_path in week8_export_files.items():

    file_exists = file_path.exists()

    file_size = (
        file_path.stat().st_size
        if file_exists
        else 0
    )

    export_validation_rows.append({
        "artifact_name": artifact_name,
        "file_name": file_path.name,
        "file_exists": file_exists,
        "file_size_bytes": file_size,
        "status": (
            "PASS"
            if file_exists and file_size > 0
            else "FAIL"
        )
    })

week8_export_validation = pd.DataFrame(
    export_validation_rows
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — FINAL DELIVERABLE EXPORT")
print("=" * 70)

print("\nOutput directory:")
print(WEEK8_OUTPUT_DIR)

print("\nExported files:")
display(week8_export_validation)

print("\nExport status counts:")
display(
    week8_export_validation[
        "status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="file_count")
)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert len(week8_export_validation) == 9

assert (
    week8_export_validation[
        "file_exists"
    ].all()
)

assert (
    week8_export_validation[
        "file_size_bytes"
    ].gt(0).all()
)

assert (
    week8_export_validation[
        "status"
    ].eq("PASS").all()
)

print(
    "\n✅ All Week 8 deliverables exported successfully."
)

WEEK 8 — FINAL DELIVERABLE EXPORT

Output directory:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week8

Exported files:


,artifact_name,file_name,file_exists,file_size_bytes,status
0,candidate_deployment_master,week8_candidate_deployment_master.csv,True,173325,PASS
1,streamlit_top5_recommendations,week8_streamlit_top5_recommendations.csv,True,74959,PASS
2,final_deployment_rag,week8_final_deployment_rag.csv,True,159167,PASS
3,occupation_deployment_summary,week8_occupation_deployment_summary.csv,True,546,PASS
4,project_kpi_summary,week8_project_kpi_summary.csv,True,566,PASS
5,final_project_insights,week8_final_project_insights.csv,True,2443,PASS
6,deployment_artifact_register,week8_deployment_artifact_register.csv,True,1192,PASS
7,deployment_configuration,week8_deployment_configuration.csv,True,473,PASS
8,integrity_validation,week8_integrity_validation.csv,True,744,PASS



Export status counts:


,status,file_count
0,PASS,9



✅ All Week 8 deliverables exported successfully.


In [55]:
# ============================================================
# WEEK 8 — CELL 17
# FINAL EXPORTED-FILE READ-BACK VALIDATION
# ============================================================

# ------------------------------------------------------------
# Reload exported CSV files from disk
# ------------------------------------------------------------

rb_candidate_master = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_candidate_deployment_master.csv"
)

rb_top5 = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_streamlit_top5_recommendations.csv"
)

rb_rag = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_final_deployment_rag.csv"
)

rb_occupation_summary = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_occupation_deployment_summary.csv"
)

rb_kpi = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_project_kpi_summary.csv"
)

rb_insights = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_final_project_insights.csv"
)

rb_artifact_register = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_deployment_artifact_register.csv"
)

rb_configuration = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_deployment_configuration.csv"
)

rb_integrity = pd.read_csv(
    WEEK8_OUTPUT_DIR / "week8_integrity_validation.csv"
)

# ------------------------------------------------------------
# Build read-back validation checks
# ------------------------------------------------------------

readback_checks = [
    {
        "check": "Candidate deployment master row count",
        "expected": 63,
        "actual": len(rb_candidate_master)
    },
    {
        "check": "Candidate deployment master unique candidates",
        "expected": 63,
        "actual": rb_candidate_master["candidate_id"].nunique()
    },
    {
        "check": "Top-5 recommendation row count",
        "expected": 315,
        "actual": len(rb_top5)
    },
    {
        "check": "Top-5 unique candidates",
        "expected": 63,
        "actual": rb_top5["candidate_id"].nunique()
    },
    {
        "check": "Final RAG row count",
        "expected": 63,
        "actual": len(rb_rag)
    },
    {
        "check": "Occupation summary row count",
        "expected": 4,
        "actual": len(rb_occupation_summary)
    },
    {
        "check": "KPI summary row count",
        "expected": 16,
        "actual": len(rb_kpi)
    },
    {
        "check": "Final insights row count",
        "expected": 10,
        "actual": len(rb_insights)
    },
    {
        "check": "Artifact register row count",
        "expected": 6,
        "actual": len(rb_artifact_register)
    },
    {
        "check": "Deployment configuration row count",
        "expected": 11,
        "actual": len(rb_configuration)
    },
    {
        "check": "Integrity audit row count",
        "expected": 12,
        "actual": len(rb_integrity)
    }
]

week8_readback_validation = pd.DataFrame(
    readback_checks
)

week8_readback_validation["status"] = np.where(
    week8_readback_validation["expected"]
    == week8_readback_validation["actual"],
    "PASS",
    "FAIL"
)

# ------------------------------------------------------------
# Additional integrity checks
# ------------------------------------------------------------

top5_exactly_five = (
    rb_top5
    .groupby("candidate_id")
    .size()
    .eq(5)
    .all()
)

all_candidates_ready = (
    rb_candidate_master[
        "deployment_status"
    ]
    .eq("Ready")
    .all()
)

all_rag_ready = (
    rb_rag[
        "final_rag_status"
    ]
    .eq("Ready")
    .all()
)

no_missing_rag = (
    rb_candidate_master[
        "final_rag_explanation"
    ]
    .notna()
    .all()
)

saved_integrity_pass = (
    rb_integrity[
        "status"
    ]
    .eq("PASS")
    .all()
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("WEEK 8 — EXPORTED-FILE READ-BACK VALIDATION")
print("=" * 70)

display(week8_readback_validation)

print("\nRead-back status counts:")

display(
    week8_readback_validation[
        "status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="check_count")
)

print("\nAdditional saved-file checks:")

print(
    "Exactly 5 recommendations per candidate:",
    top5_exactly_five
)

print(
    "All deployment candidates ready:",
    all_candidates_ready
)

print(
    "All RAG explanations ready:",
    all_rag_ready
)

print(
    "No missing RAG explanations:",
    no_missing_rag
)

print(
    "Saved integrity audit contains only PASS:",
    saved_integrity_pass
)

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert (
    week8_readback_validation[
        "status"
    ].eq("PASS").all()
)

assert top5_exactly_five
assert all_candidates_ready
assert all_rag_ready
assert no_missing_rag
assert saved_integrity_pass

print(
    "\n✅ WEEK 8 FINAL VALIDATION PASSED."
)

print(
    "✅ All exported deployment artifacts were "
    "successfully reloaded and verified."
)

WEEK 8 — EXPORTED-FILE READ-BACK VALIDATION


,check,expected,actual,status
0,Candidate deployment master row count,63,63,PASS
1,Candidate deployment master unique candidates,63,63,PASS
2,Top-5 recommendation row count,315,315,PASS
3,Top-5 unique candidates,63,63,PASS
4,Final RAG row count,63,63,PASS
5,Occupation summary row count,4,4,PASS
6,KPI summary row count,16,16,PASS
7,Final insights row count,10,10,PASS
8,Artifact register row count,6,6,PASS
9,Deployment configuration row count,11,11,PASS



Read-back status counts:


,status,check_count
0,PASS,11



Additional saved-file checks:
Exactly 5 recommendations per candidate: True
All deployment candidates ready: True
All RAG explanations ready: True
No missing RAG explanations: True
Saved integrity audit contains only PASS: True

✅ WEEK 8 FINAL VALIDATION PASSED.
✅ All exported deployment artifacts were successfully reloaded and verified.
